# GreenSM Data Centric Challenge: notebook mẫu

Notebook này là **khung mẫu đơn giản**: bạn nạp dữ liệu, huấn luyện YOLOv8n, rồi xuất ONNX và đóng gói bài nộp.
Bạn được **tự do thay đổi phần dữ liệu và phần huấn luyện** (Bước 1 và 2). **Bước 3 (xuất, kiểm tra, đóng gói) là bắt buộc và không sửa**:
nó dùng đúng bộ kiểm tra của hệ thống chấm, không đạt thì không tạo `submission.zip`.

| Bước | Việc làm | Bạn được sửa? |
|---|---|---|
| 0 | Cài thư viện, điền **mã số đội** (cũng là seed) | Chỉ điền mã số đội |
| 1 | Nạp dữ liệu YOLO của đội (zip) | Có, tự do |
| 2 | Huấn luyện YOLOv8n (kèm gợi ý kỹ thuật nâng cao) | Có, tự do |
| 3 | **Xuất ONNX, kiểm tra, đóng gói `submission.zip`, tải về** | **Không** |

**Cố định:** mô hình `yolov8n.pt` (kiến trúc YOLOv8n), `imgsz=640`, **một lớp duy nhất `GreenSM`** (id 0), tham số xuất ONNX do ban tổ chức quy định.

**Cách chạy:** bật GPU (Runtime > Change runtime type > **T4 GPU**), rồi chạy lần lượt từng ô từ trên xuống.

**Giữ lại** notebook này và zip dữ liệu của đội: ban tổ chức có thể yêu cầu chạy lại để tái lập kết quả (cùng dữ liệu, cùng mã số đội).

In [ ]:
#@title Hằng số của cuộc thi (tự sinh từ hệ thống chấm, không chỉnh sửa) { display-mode: "form" }
# GENERATED by notebook/simple/build_simple_notebook.py from competition_config.yaml, detscore/data/*.json and detscore/*.py.
# Do not edit by hand: change the sources and rebuild, so the notebook cannot drift from the server.
import json

NOTEBOOK_NAME = 'train_and_export'
NOTEBOOK_VERSION = '2.0.0+c4ac445c'
DETSCORE_VERSION = '0.1.0'
ULTRALYTICS_VERSION = '8.4.170'
INSTALL_SPECS = ['ultralytics==8.4.170', 'onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.82', 'onnxruntime']
MODEL_NAME = 'yolov8n.pt'
IMGSZ = 640
CLASSES = ['GreenSM']
CLASS_DISPLAY_VI = {'GreenSM': 'GreenSM'}
EXPORT_CONTRACT = json.loads(r'''{
 "batch": 1,
 "dynamic": false,
 "export_call": "YOLO(best_pt).export(format='onnx', imgsz=640, opset=17, simplify=True, dynamic=False, half=False, nms=False, batch=1)",
 "format": "onnx",
 "half": false,
 "imgsz": 640,
 "input": {
  "dtype": "float32",
  "name": "images",
  "shape": [
   1,
   3,
   640,
   640
  ]
 },
 "nms": false,
 "notes": "Class names and nc come from competition_config.yaml, never from this file.",
 "opset": 17,
 "output": {
  "dtype": "float32",
  "name": "output0",
  "shape_template": [
   1,
   "4+nc",
   8400
  ]
 },
 "rationale": {
  "half": "False. ultralytics >= 8.4 deprecates half= in favour of quantize=; half=False is still accepted at the pinned version and means FP32.",
  "nms": "False. For YOLOv8 (no NMS-free head) this keeps NMS external; the host applies class-aware NMS itself, so the ONNX output is the raw [1,4+nc,8400] tensor.",
  "opset": "Pinned to 17: without it ultralytics derives the opset from the torch version, so Colab and other machines could silently differ. 17 loads in every onnxruntime >= 1.14.",
  "simplify": "True (onnxslim): folds constant sub-graphs, giving a smaller and more stable graph across torch/onnx versions than the raw trace. Ultralytics default."
 },
 "schema_version": 1,
 "simplify": true,
 "structure_reference": {
  "allowed_op_types": [
   "Add",
   "Concat",
   "Constant",
   "Conv",
   "Div",
   "Gather",
   "Identity",
   "MaxPool",
   "Mul",
   "Reshape",
   "Resize",
   "Shape",
   "Sigmoid",
   "Slice",
   "Softmax",
   "Split",
   "Sub",
   "Transpose"
  ],
  "allowed_op_types_note": "Union over the contract export and the simplify=False / opset-12 variants. Custom domains are never allowed.",
  "contract_export_op_types": {
   "Add": 8,
   "Concat": 17,
   "Conv": 64,
   "Div": 1,
   "MaxPool": 3,
   "Mul": 58,
   "Reshape": 8,
   "Resize": 2,
   "Sigmoid": 58,
   "Slice": 2,
   "Softmax": 1,
   "Split": 8,
   "Sub": 2,
   "Transpose": 1
  },
  "conv_nodes": 64,
  "conv_signature_definition": "sha256(repr(sorted((role, dtype, shape) for every Conv weight (role 0) and bias (role 1)))) with tensors resolved through Identity nodes; no names, no values. Independent of simplify, opset (12 and 17 tested) and tensor naming; differs for nc, architecture and fp16.",
  "conv_signature_sha256_nc1": "33bcc5aa54b9b539e91ecd793ffcc273b8f27643bd400458c20337f5fb288697",
  "duplicate_fingerprint_note": "The spec fingerprint (name+dtype+shape+bytes of all initializers) equals between re-exports of the same .pt with the same toolchain but differs for simplify/opset variants of the same weights. The nameless Conv-content fingerprint (conv_content_fingerprint in manifest.json) is equal for all of them and is the stronger copy detector.",
  "status": "PROVISIONAL: derived from the local seeded export (fixture good.onnx). The TA regenerates it once from a known-good Colab export: `scripts/make_fixtures.sh --describe model.onnx` and `--compare .fixtures/onnx/good.onnx model.onnx`."
 },
 "ultralytics_version": "8.4.170"
}''')
SELFCHECK_BUNDLE = json.loads(r'''{
 "config_yaml": "# Subset of competition_config.yaml (generated by build_notebook.py)\nschema_version: 1\nclasses:\n- GreenSM\nclass_display_vi:\n  GreenSM: GreenSM\nimgsz: 640\npostprocess:\n  conf: 0.001\n  iou: 0.7\n  max_det: 300\n  pad_value: 114\nsubmission:\n  required_entry: model.onnx\n  max_zip_mb: 25\n  max_uncompressed_mb: 50\n  max_compression_ratio: 20\n  ratio_check_min_mb: 1\n  max_zip_entries_scanned: 100\n  max_central_directory_kb: 64\nvalidator:\n  max_nodes: 2000\n  max_initializers: 1500\n  max_total_params_m: 30\n  max_metadata_chars: 20000\nsplit:\n  public_images: 40\n  private_images: 80\n  min_gt_per_class_per_split: 30\n",
 "files": {
  "__init__.py": "\"\"\"detscore: submission validator, ONNX runner, decoder/NMS and mAP scorer.\"\"\"\n\n__version__ = \"0.1.0\"\n",
  "config.py": "\"\"\"Competition configuration loader.\n\n``competition_config.yaml`` is the single source of truth for classes, thresholds and policies.\nResolution order for the file: explicit ``path`` argument, environment variable ``DETSCORE_CONFIG``,\n``<project root>/competition_config.yaml`` (source checkout), ``./competition_config.yaml``.\n\n``Config`` exposes the values other modules need as plain attributes (``classes``, ``nc``, ``imgsz``,\n``conf``, ``iou``, ``max_det``, ``num_anchors``, ``max_zip_bytes``, ``max_uncompressed_bytes``,\n``penalty_points`` ...). The raw parsed mapping is kept in ``Config.raw`` for sections this module does\nnot interpret (for example ``competitions`` and ``container``).\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\nimport yaml\n\nENV_VAR = \"DETSCORE_CONFIG\"\nCONFIG_FILENAME = \"competition_config.yaml\"\nSTRIDES = (8, 16, 32)  # YOLOv8 detection strides: anchors = sum((imgsz / s) ** 2)\n\n_MB = 1024 * 1024\n\n\nclass ConfigError(ValueError):\n    \"\"\"The competition configuration is missing or inconsistent.\"\"\"\n\n\ndef anchors_for(imgsz: int) -> int:\n    \"\"\"Number of YOLOv8 anchor points for a square input (8400 for 640).\"\"\"\n    if imgsz % max(STRIDES):\n        raise ConfigError(f\"imgsz must be a multiple of {max(STRIDES)}, got {imgsz}\")\n    return sum((imgsz // s) ** 2 for s in STRIDES)\n\n\ndef _section(raw: Mapping[str, Any], key: str) -> Mapping[str, Any]:\n    value = raw.get(key, {})\n    if not isinstance(value, Mapping):\n        raise ConfigError(f\"'{key}' must be a mapping\")\n    return value\n\n\ndef _num(section: Mapping[str, Any], key: str, default: float | None = None, *, minimum: float = 0.0) -> float:\n    if key not in section and default is None:\n        raise ConfigError(f\"missing required key '{key}'\")\n    value = section.get(key, default)\n    if isinstance(value, bool) or not isinstance(value, (int, float)):\n        raise ConfigError(f\"'{key}' must be a number, got {value!r}\")\n    if value < minimum:\n        raise ConfigError(f\"'{key}' must be >= {minimum}, got {value!r}\")\n    return value\n\n\n@dataclass(frozen=True)\nclass Config:\n    classes: tuple[str, ...]\n    class_display_vi: Mapping[str, str]\n    imgsz: int\n    # post-processing\n    conf: float\n    iou: float\n    max_det: int\n    pad_value: int\n    # submission zip\n    required_entry: str\n    max_zip_bytes: int\n    max_uncompressed_bytes: int\n    max_compression_ratio: float\n    ratio_check_min_bytes: int\n    max_zip_entries_scanned: int\n    max_central_directory_bytes: int\n    # onnx validator guards\n    max_nodes: int\n    max_initializers: int\n    max_total_params: int\n    max_metadata_chars: int\n    # scoring data\n    split_public_images: int\n    split_private_images: int\n    min_gt_per_class_per_split: int\n    # penalties\n    penalty_points: float\n    original_extra_slots: int\n    copier_extra_slots: int\n    # accounts\n    username_pattern: str\n    # container\n    validate_timeout_s: int\n    infer_timeout_s: int\n    container_cpus: float\n    container_memory_mb: int\n    # untouched sections\n    competitions: tuple[Mapping[str, Any], ...] = ()\n    raw: Mapping[str, Any] = field(default_factory=dict, compare=False, repr=False)\n    source: str = field(default=\"<dict>\", compare=False)\n    # container hardening (defaults keep Config(...) built by older callers valid)\n    container_pids_limit: int = 64\n    container_tmpfs_mb: int = 256\n\n    @property\n    def nc(self) -> int:\n        return len(self.classes)\n\n    @property\n    def num_anchors(self) -> int:\n        return anchors_for(self.imgsz)\n\n    @property\n    def max_zip_mb(self) -> float:\n        return self.max_zip_bytes / _MB\n\n    @property\n    def max_uncompressed_mb(self) -> float:\n        return self.max_uncompressed_bytes / _MB\n\n    @property\n    def expected_input_shape(self) -> tuple[int, int, int, int]:\n        return (1, 3, self.imgsz, self.imgsz)\n\n    @property\n    def expected_output_shape(self) -> tuple[int, int, int]:\n        return (1, 4 + self.nc, self.num_anchors)\n\n    @classmethod\n    def from_dict(cls, raw: Mapping[str, Any], source: str = \"<dict>\") -> \"Config\":\n        if not isinstance(raw, Mapping):\n            raise ConfigError(\"the configuration root must be a mapping\")\n        classes = raw.get(\"classes\")\n        if not isinstance(classes, (list, tuple)) or not classes:\n            raise ConfigError(\"'classes' must be a non-empty list\")\n        if not all(isinstance(c, str) and c for c in classes):\n            raise ConfigError(\"every class name must be a non-empty string\")\n        if len(set(classes)) != len(classes):\n            raise ConfigError(\"class names must be unique\")\n        display = dict(_section(raw, \"class_display_vi\"))\n        unknown = set(display) - set(classes)\n        if unknown:\n            raise ConfigError(f\"class_display_vi names unknown classes: {sorted(unknown)}\")\n        display = {c: str(display.get(c, c)) for c in classes}\n\n        imgsz = int(_num(raw, \"imgsz\", 640, minimum=32))\n        anchors_for(imgsz)  # validates the multiple-of-32 constraint\n\n        post = _section(raw, \"postprocess\")\n        sub = _section(raw, \"submission\")\n        val = _section(raw, \"validator\")\n        split = _section(raw, \"split\")\n        pen = _section(raw, \"penalty\")\n        auth = _section(raw, \"auth\")\n        cont = _section(raw, \"container\")\n\n        conf = float(_num(post, \"conf\", 0.001))\n        iou = float(_num(post, \"iou\", 0.7))\n        if not (0 <= conf <= 1 and 0 < iou <= 1):\n            raise ConfigError(\"postprocess.conf must be in [0,1] and postprocess.iou in (0,1]\")\n\n        comps = raw.get(\"competitions\", [])\n        if not isinstance(comps, (list, tuple)):\n            raise ConfigError(\"'competitions' must be a list\")\n        codes = [c.get(\"code\") for c in comps if isinstance(c, Mapping)]\n        if len(codes) != len(comps) or len(set(codes)) != len(codes) or not all(isinstance(c, str) and c for c in codes):\n            raise ConfigError(\"every competition needs a unique string 'code'\")\n\n        return cls(\n            classes=tuple(classes),\n            class_display_vi=display,\n            imgsz=imgsz,\n            conf=conf,\n            iou=iou,\n            max_det=int(_num(post, \"max_det\", 300, minimum=1)),\n            pad_value=int(_num(post, \"pad_value\", 114)),\n            required_entry=str(sub.get(\"required_entry\", \"model.onnx\")),\n            max_zip_bytes=int(_num(sub, \"max_zip_mb\", 25, minimum=0.001) * _MB),\n            max_uncompressed_bytes=int(_num(sub, \"max_uncompressed_mb\", 50, minimum=0.001) * _MB),\n            max_compression_ratio=float(_num(sub, \"max_compression_ratio\", 20, minimum=1)),\n            ratio_check_min_bytes=int(_num(sub, \"ratio_check_min_mb\", 1) * _MB),\n            max_zip_entries_scanned=int(_num(sub, \"max_zip_entries_scanned\", 100, minimum=1)),\n            max_central_directory_bytes=int(_num(sub, \"max_central_directory_kb\", 64, minimum=1) * 1024),\n            max_nodes=int(_num(val, \"max_nodes\", 2000, minimum=1)),\n            max_initializers=int(_num(val, \"max_initializers\", 1500, minimum=1)),\n            max_total_params=int(_num(val, \"max_total_params_m\", 30, minimum=0.001) * 1_000_000),\n            max_metadata_chars=int(_num(val, \"max_metadata_chars\", 20000, minimum=1)),\n            split_public_images=int(_num(split, \"public_images\", 40, minimum=1)),\n            split_private_images=int(_num(split, \"private_images\", 80, minimum=1)),\n            min_gt_per_class_per_split=int(_num(split, \"min_gt_per_class_per_split\", 30)),\n            penalty_points=float(_num(pen, \"points\", 10)),\n            original_extra_slots=int(_num(pen, \"original_extra_slots\", 1)),\n            copier_extra_slots=int(_num(pen, \"copier_extra_slots\", 1)),\n            username_pattern=str(auth.get(\"username_pattern\", r\"^[a-z0-9_.-]{3,64}$\")),\n            validate_timeout_s=int(_num(cont, \"validate_timeout_s\", 60, minimum=1)),\n            infer_timeout_s=int(_num(cont, \"infer_timeout_s\", 120, minimum=1)),\n            container_cpus=float(_num(cont, \"cpus\", 2, minimum=0.1)),\n            container_memory_mb=int(_num(cont, \"memory_mb\", 2048, minimum=64)),\n            container_pids_limit=int(_num(cont, \"pids_limit\", 64, minimum=8)),\n            container_tmpfs_mb=int(_num(cont, \"tmpfs_mb\", 256, minimum=16)),\n            competitions=tuple(dict(c) for c in comps),\n            raw=dict(raw),\n            source=source,\n        )\n\n\ndef default_search_paths() -> list[Path]:\n    paths: list[Path] = []\n    env = os.environ.get(ENV_VAR)\n    if env:\n        paths.append(Path(env))\n    # <root>/detscore/detscore/config.py -> <root>\n    paths.append(Path(__file__).resolve().parents[2] / CONFIG_FILENAME)\n    paths.append(Path.cwd() / CONFIG_FILENAME)\n    return paths\n\n\ndef load_config(path: str | os.PathLike | None = None) -> Config:\n    \"\"\"Load and validate the configuration (see the module docstring for the lookup order).\"\"\"\n    if path is not None:\n        candidates = [Path(path)]\n    else:\n        candidates = default_search_paths()\n    for candidate in candidates:\n        if candidate.is_file():\n            try:\n                raw = yaml.safe_load(candidate.read_text(encoding=\"utf-8\"))\n            except yaml.YAMLError as exc:\n                raise ConfigError(f\"{candidate}: invalid YAML: {exc}\") from exc\n            try:\n                return Config.from_dict(raw, source=str(candidate))\n            except ConfigError as exc:\n                raise ConfigError(f\"{candidate}: {exc}\") from exc\n    tried = \", \".join(str(c) for c in candidates)\n    raise ConfigError(f\"{CONFIG_FILENAME} not found (looked in: {tried}); set {ENV_VAR}\")\n",
  "data/reference_signature.json": "{\n  \"schema_version\": 1,\n  \"nc\": 1,\n  \"imgsz\": 640,\n  \"signature_sha256\": \"33bcc5aa54b9b539e91ecd793ffcc273b8f27643bd400458c20337f5fb288697\",\n  \"conv_nodes\": 64,\n  \"conv_params\": 3000448,\n  \"conv_tensors\": [\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        1,\n        16,\n        1,\n        1\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        1,\n        64,\n        1,\n        1\n      ],\n      \"count\": 3\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        16,\n        3,\n        3,\n        3\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        16,\n        16,\n        3,\n        3\n      ],\n      \"count\": 2\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        32,\n        16,\n        3,\n        3\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        32,\n        32,\n        1,\n        1\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        32,\n        32,\n        3,\n        3\n      ],\n      \"count\": 6\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        32,\n        48,\n        1,\n        1\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        64,\n        32,\n        3,\n        3\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        64,\n        64,\n        1,\n        1\n      ],\n      \"count\": 4\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        64,\n        64,\n        3,\n        3\n      ],\n      \"count\": 17\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        64,\n        96,\n        1,\n        1\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        64,\n        128,\n        1,\n        1\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        64,\n        128,\n        3,\n        3\n      ],\n      \"count\": 2\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        64,\n        192,\n        1,\n        1\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        64,\n        256,\n        3,\n        3\n      ],\n      \"count\": 2\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        128,\n        64,\n        3,\n        3\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        128,\n        128,\n        1,\n        1\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        128,\n        128,\n        3,\n        3\n      ],\n      \"count\": 5\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        128,\n        192,\n        1,\n        1\n      ],\n      \"count\": 3\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        128,\n        256,\n        1,\n        1\n      ],\n      \"count\": 2\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        128,\n        384,\n        1,\n        1\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        256,\n        128,\n        3,\n        3\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        256,\n        256,\n        1,\n        1\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        256,\n        384,\n        1,\n        1\n      ],\n      \"count\": 3\n    },\n    {\n      \"role\": 0,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        256,\n        512,\n        1,\n        1\n      ],\n      \"count\": 1\n    },\n    {\n      \"role\": 1,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        1\n      ],\n      \"count\": 3\n    },\n    {\n      \"role\": 1,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        16\n      ],\n      \"count\": 3\n    },\n    {\n      \"role\": 1,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        32\n      ],\n      \"count\": 9\n    },\n    {\n      \"role\": 1,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        64\n      ],\n      \"count\": 29\n    },\n    {\n      \"role\": 1,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        128\n      ],\n      \"count\": 13\n    },\n    {\n      \"role\": 1,\n      \"dtype\": \"float32\",\n      \"shape\": [\n        256\n      ],\n      \"count\": 6\n    }\n  ],\n  \"allowed_op_types\": [\n    \"Add\",\n    \"Concat\",\n    \"Constant\",\n    \"Conv\",\n    \"Div\",\n    \"Gather\",\n    \"Identity\",\n    \"MaxPool\",\n    \"Mul\",\n    \"Reshape\",\n    \"Resize\",\n    \"Shape\",\n    \"Sigmoid\",\n    \"Slice\",\n    \"Softmax\",\n    \"Split\",\n    \"Sub\",\n    \"Transpose\"\n  ],\n  \"reference_op_counts\": {\n    \"Add\": 8,\n    \"Concat\": 17,\n    \"Conv\": 64,\n    \"Div\": 1,\n    \"MaxPool\": 3,\n    \"Mul\": 58,\n    \"Reshape\": 8,\n    \"Resize\": 2,\n    \"Sigmoid\": 58,\n    \"Slice\": 2,\n    \"Softmax\": 1,\n    \"Split\": 8,\n    \"Sub\": 2,\n    \"Transpose\": 1\n  },\n  \"provenance\": {\n    \"generated_at\": \"2026-10-03T06:23:13+00:00\",\n    \"note\": \"local seeded random-init export (fixtures), nc=1 GreenSM\",\n    \"source\": \"onnx\",\n    \"primary_file_sha256\": \"9b8a907ac1ce2bb79e6ee2eb6170389d4456fab06dd5a1b05ce51979d8b61454\",\n    \"primary_file\": \"good.onnx\",\n    \"variant_files\": [\n      \"good_variant_nosimplify.onnx\",\n      \"good_variant_opset_alt.onnx\"\n    ],\n    \"variants_merged\": 2,\n    \"status\": \"PROVISIONAL: built from a local export; the TA should regenerate it once from a known-good Colab export (--onnx colab_model.onnx) and check it with tools/compare_reference.py\"\n  }\n}\n",
  "data/reserved_fingerprints.json": "{\n  \"schema_version\": 1,\n  \"generated_at\": \"2026-10-01T05:37:51+00:00\",\n  \"entries\": [\n    {\n      \"kind\": \"pretrained\",\n      \"label\": \"Ultralytics YOLOv8n COCO pretrained (yolov8n.pt)\",\n      \"tensor_fingerprint\": \"7d9656a7f9fe0ffabbb7aaeb4789c37636e804293aa979c677a3f97e27e8309d\",\n      \"content_fingerprint\": \"818956524d709638c751f53fcf92f3211e09aa5d96f82aa8f186eabe49e02115\",\n      \"conv_signature_sha256\": \"278ee59eeb2055f26cab36b6ff1a2acbcbe3d49ee663af2960b15a8a62c644ea\",\n      \"file_sha256\": \"4f5ed7480e0e2271caa919dd747cd5b9bee1601d1d68f1b6391e8121f48774e3\",\n      \"source_file\": \"pretrained_nc80.onnx\"\n    }\n  ]\n}\n",
  "errors.py": "\"\"\"Submission errors: a fixed code table and accented-Vietnamese messages.\n\nEvery message names the exact problem and the fix. The factory functions below are the only place where\nstudent-facing validator text lives. No error here consumes a submission slot: an invalid submission is\nrejected and the team may try again (instructor clarification 1).\n\"\"\"\nfrom __future__ import annotations\n\nfrom typing import Any, Iterable\n\n# code -> (stage that produces it, short Vietnamese title)\nCODE_TABLE: dict[str, tuple[str, str]] = {\n    \"BAD_EXTENSION\": (\"upload\", \"Sai \u0111\u1ecbnh d\u1ea1ng t\u1ec7p\"),\n    \"ZIP_CORRUPT\": (\"upload\", \"T\u1ec7p zip b\u1ecb h\u1ecfng\"),\n    \"ZIP_TOO_LARGE\": (\"upload\", \"T\u1ec7p qu\u00e1 l\u1edbn\"),\n    \"ZIP_BOMB\": (\"upload\", \"T\u1ec7p n\u00e9n v\u01b0\u1ee3t gi\u1edbi h\u1ea1n gi\u1ea3i n\u00e9n\"),\n    \"ZIP_ENTRIES\": (\"upload\", \"C\u1ea5u tr\u00fac zip kh\u00f4ng \u0111\u00fang\"),\n    \"PATH_TRAVERSAL\": (\"upload\", \"\u0110\u01b0\u1eddng d\u1eabn kh\u00f4ng h\u1ee3p l\u1ec7 trong zip\"),\n    \"ONNX_PARSE\": (\"validate\", \"Kh\u00f4ng \u0111\u1ecdc \u0111\u01b0\u1ee3c t\u1ec7p ONNX\"),\n    \"ONNX_EXTERNAL_DATA\": (\"validate\", \"ONNX d\u00f9ng d\u1eef li\u1ec7u ngo\u00e0i\"),\n    \"ONNX_CUSTOM_OP\": (\"validate\", \"ONNX c\u00f3 to\u00e1n t\u1eed t\u00f9y bi\u1ebfn\"),\n    \"ONNX_IO\": (\"validate\", \"Input/output ONNX sai\"),\n    \"ONNX_DTYPE\": (\"validate\", \"Sai ki\u1ec3u d\u1eef li\u1ec7u\"),\n    \"STRUCTURE_MISMATCH\": (\"validate\", \"Ki\u1ebfn tr\u00fac m\u00f4 h\u00ecnh kh\u00f4ng kh\u1edbp\"),\n    \"META_NAMES\": (\"validate\", \"T\u00ean l\u1edbp trong metadata sai\"),\n    \"PRETRAINED_MODEL\": (\"validate\", \"M\u00f4 h\u00ecnh pretrained g\u1ed1c\"),\n    \"BASELINE_MODEL\": (\"validate\", \"M\u00f4 h\u00ecnh baseline \u0111\u01b0\u1ee3c cung c\u1ea5p\"),\n    \"INVALID_WEIGHTS\": (\"validate\", \"Tr\u1ecdng s\u1ed1 m\u00f4 h\u00ecnh h\u1ecfng\"),\n    \"TIME_LIMIT\": (\"infer\", \"Qu\u00e1 th\u1eddi gian ch\u1ea1y\"),\n    \"MEMORY_LIMIT\": (\"infer\", \"Qu\u00e1 gi\u1edbi h\u1ea1n b\u1ed9 nh\u1edb\"),\n    \"RUNTIME_ERROR\": (\"infer\", \"L\u1ed7i khi ch\u1ea1y m\u00f4 h\u00ecnh\"),\n}\nCODES: tuple[str, ...] = tuple(CODE_TABLE)\n\n# Every code is a rejection that does NOT consume quota.\nCHARGES_QUOTA = False\n\n\nclass SubmissionError(Exception):\n    \"\"\"A submission is invalid. ``message_vi`` is shown to the team; ``detail`` is for logs/admins.\"\"\"\n\n    def __init__(self, code: str, message_vi: str, detail: dict[str, Any] | None = None):\n        if code not in CODE_TABLE:\n            raise ValueError(f\"unknown submission error code: {code!r}\")\n        super().__init__(f\"{code}: {message_vi}\")\n        self.code = code\n        self.message_vi = message_vi\n        self.detail: dict[str, Any] = dict(detail or {})\n\n    @property\n    def stage(self) -> str:\n        return CODE_TABLE[self.code][0]\n\n    @property\n    def title_vi(self) -> str:\n        return CODE_TABLE[self.code][1]\n\n    def to_dict(self) -> dict[str, Any]:\n        return {\"code\": self.code, \"message_vi\": self.message_vi, \"detail\": self.detail}\n\n\ndef _mb(n: float) -> str:\n    return f\"{n / (1024 * 1024):.1f} MB\"\n\n\ndef _list(items: Iterable[str], limit: int = 6) -> str:\n    items = [str(i) for i in items]\n    shown = \", \".join(f\"\u00ab{i}\u00bb\" for i in items[:limit])\n    if len(items) > limit:\n        shown += f\" v\u00e0 {len(items) - limit} m\u1ee5c kh\u00e1c\"\n    return shown\n\n\ndef _shape(dims: Iterable[Any]) -> str:\n    return \"[\" + \", \".join(str(d) for d in dims) + \"]\"\n\n\n# ---------------------------------------------------------------------------------------------- upload layer\ndef bad_extension(filename: str, required: str = \".zip\") -> SubmissionError:\n    name = filename or \"(kh\u00f4ng c\u00f3 t\u00ean)\"\n    low = name.lower()\n    ext = low[low.rfind(\".\"):] if \".\" in low else \"(kh\u00f4ng c\u00f3 \u0111u\u00f4i)\"\n    msg = f\"T\u1ec7p \u00ab{name}\u00bb c\u00f3 \u0111u\u00f4i \u00ab{ext}\u00bb, nh\u01b0ng h\u1ec7 th\u1ed1ng ch\u1ec9 nh\u1eadn t\u1ec7p {required} ch\u1ee9a \u0111\u00fang m\u1ed9t t\u1ec7p model.onnx.\"\n    if ext == \".onnx\":\n        msg += \" B\u1ea1n \u0111ang n\u1ed9p tr\u1ef1c ti\u1ebfp t\u1ec7p .onnx: h\u00e3y n\u00e9n n\u00f3 th\u00e0nh submission.zip (t\u00ean b\u00ean trong zip ph\u1ea3i l\u00e0 model.onnx) r\u1ed3i n\u1ed9p l\u1ea1i.\"\n    elif ext in (\".pt\", \".pth\"):\n        msg += \" \u0110\u00e2y l\u00e0 checkpoint PyTorch: h\u00e3y export sang ONNX (\u00f4 export trong notebook), r\u1ed3i n\u00e9n model.onnx th\u00e0nh zip.\"\n    else:\n        msg += \" H\u00e3y d\u00f9ng \u00f4 cu\u1ed1i c\u1ee7a notebook \u0111\u1ec3 t\u1ea1o submission.zip r\u1ed3i n\u1ed9p l\u1ea1i.\"\n    return SubmissionError(\"BAD_EXTENSION\", msg, {\"filename\": filename, \"extension\": ext})\n\n\ndef zip_corrupt(reason: str) -> SubmissionError:\n    return SubmissionError(\n        \"ZIP_CORRUPT\",\n        f\"T\u1ec7p zip b\u1ecb h\u1ecfng ho\u1eb7c kh\u00f4ng \u0111\u1ecdc \u0111\u01b0\u1ee3c ({reason}). C\u00f3 th\u1ec3 qu\u00e1 tr\u00ecnh t\u1ea3i l\u00ean b\u1ecb ng\u1eaft gi\u1eefa ch\u1eebng ho\u1eb7c t\u1ec7p kh\u00f4ng ph\u1ea3i zip: \"\n        \"h\u00e3y t\u1ea1o l\u1ea1i submission.zip (ch\u1ea1y l\u1ea1i \u00f4 cu\u1ed1i c\u1ee7a notebook), ki\u1ec3m tra dung l\u01b0\u1ee3ng t\u1ec7p r\u1ed3i n\u1ed9p l\u1ea1i.\",\n        {\"reason\": reason},\n    )\n\n\ndef zip_too_large(size_bytes: int, limit_bytes: int, what: str = \"T\u1ec7p zip\") -> SubmissionError:\n    return SubmissionError(\n        \"ZIP_TOO_LARGE\",\n        f\"{what} n\u1eb7ng {_mb(size_bytes)}, v\u01b0\u1ee3t gi\u1edbi h\u1ea1n {_mb(limit_bytes)}. \"\n        \"M\u00f4 h\u00ecnh YOLOv8n fp32 ch\u1ec9 kho\u1ea3ng 12 MB: ki\u1ec3m tra b\u1ea1n kh\u00f4ng export nh\u1ea7m YOLOv8s/m/l/x v\u00e0 kh\u00f4ng nh\u00e9t th\u00eam t\u1ec7p n\u00e0o v\u00e0o zip.\",\n        {\"size_bytes\": size_bytes, \"limit_bytes\": limit_bytes},\n    )\n\n\ndef zip_bomb(reason: str, **detail: Any) -> SubmissionError:\n    return SubmissionError(\n        \"ZIP_BOMB\",\n        f\"T\u1ec7p zip b\u1ecb t\u1eeb ch\u1ed1i v\u00ec gi\u1ea3i n\u00e9n v\u01b0\u1ee3t gi\u1edbi h\u1ea1n an to\u00e0n ({reason}). \"\n        \"M\u1ed9t t\u1ec7p model.onnx YOLOv8n b\u00ecnh th\u01b0\u1eddng ch\u1ec9 kho\u1ea3ng 12 MB sau khi gi\u1ea3i n\u00e9n: h\u00e3y t\u1ea1o l\u1ea1i zip t\u1eeb t\u1ec7p ONNX do notebook export.\",\n        {\"reason\": reason, **detail},\n    )\n\n\ndef zip_entries_empty() -> SubmissionError:\n    return SubmissionError(\n        \"ZIP_ENTRIES\",\n        \"T\u1ec7p zip r\u1ed7ng. Zip ph\u1ea3i ch\u1ee9a \u0111\u00fang m\u1ed9t t\u1ec7p t\u00ean model.onnx (kh\u00f4ng c\u00f3 th\u01b0 m\u1ee5c, kh\u00f4ng c\u00f3 t\u1ec7p kh\u00e1c).\",\n        {\"entries\": []},\n    )\n\n\ndef zip_entries_too_many(count: int, limit: int) -> SubmissionError:\n    return SubmissionError(\n        \"ZIP_ENTRIES\",\n        f\"T\u1ec7p zip c\u00f3 {count} m\u1ee5c (gi\u1edbi h\u1ea1n \u0111\u1ecdc l\u00e0 {limit}). Zip ph\u1ea3i ch\u1ee9a \u0111\u00fang m\u1ed9t t\u1ec7p t\u00ean model.onnx v\u00e0 kh\u00f4ng c\u00f3 g\u00ec kh\u00e1c: \"\n        \"h\u00e3y t\u1ea1o l\u1ea1i zip ch\u1ec9 v\u1edbi model.onnx.\",\n        {\"count\": count, \"limit\": limit},\n    )\n\n\ndef zip_entries_wrong(names: list[str], required: str) -> SubmissionError:\n    \"\"\"Zip is not exactly one entry called ``required`` at the root; names the precise reason and fix.\"\"\"\n    detail = {\"entries\": names, \"required\": required}\n    count = len(names)\n    present = required in names\n    if present and count > 1:\n        extra = [n for n in names if n != required] if names.count(required) == 1 else names\n        if names.count(required) > 1:\n            msg = f\"Zip ch\u1ee9a {names.count(required)} m\u1ee5c c\u00f9ng t\u00ean \u00ab{required}\u00bb. Zip ph\u1ea3i ch\u1ee9a \u0111\u00fang m\u1ed9t t\u1ec7p {required}.\"\n        else:\n            msg = f\"Zip c\u00f3 {count} m\u1ee5c, nh\u01b0ng ch\u1ec9 \u0111\u01b0\u1ee3c ph\u00e9p c\u00f3 \u0111\u00fang m\u1ed9t t\u1ec7p {required}. H\u00e3y x\u00f3a c\u00e1c m\u1ee5c th\u1eeba: {_list(extra)}.\"\n        return SubmissionError(\"ZIP_ENTRIES\", msg, detail)\n    if count == 1:\n        only = names[0]\n        if only.endswith(\"/\"):\n            msg = f\"Zip ch\u1ec9 ch\u1ee9a m\u1ed9t th\u01b0 m\u1ee5c \u00ab{only}\u00bb m\u00e0 kh\u00f4ng c\u00f3 t\u1ec7p {required}. H\u00e3y n\u00e9n t\u1ec7p {required} tr\u1ef1c ti\u1ebfp, kh\u00f4ng n\u00e9n c\u1ea3 th\u01b0 m\u1ee5c.\"\n        elif only.replace(\"\\\\\", \"/\").split(\"/\")[-1] == required:\n            folder = only.replace(\"\\\\\", \"/\").rsplit(\"/\", 1)[0]\n            msg = (f\"T\u1ec7p {required} \u0111ang n\u1eb1m trong th\u01b0 m\u1ee5c \u00ab{folder}/\u00bb (\u0111\u01b0\u1eddng d\u1eabn \u00ab{only}\u00bb). \"\n                   f\"N\u00f3 ph\u1ea3i n\u1eb1m ngay \u1edf g\u1ed1c zip: h\u00e3y n\u00e9n ri\u00eang t\u1ec7p {required}, kh\u00f4ng n\u00e9n c\u1ea3 th\u01b0 m\u1ee5c.\")\n        elif only.lower() == required.lower():\n            msg = f\"T\u00ean t\u1ec7p trong zip l\u00e0 \u00ab{only}\u00bb, c\u1ea7n \u0111\u00fang \u00ab{required}\u00bb (ph\u00e2n bi\u1ec7t hoa/th\u01b0\u1eddng).\"\n        else:\n            msg = f\"T\u00ean t\u1ec7p trong zip l\u00e0 \u00ab{only}\u00bb, nh\u01b0ng b\u1eaft bu\u1ed9c ph\u1ea3i l\u00e0 \u00ab{required}\u00bb. H\u00e3y \u0111\u1ed5i t\u00ean t\u1ec7p r\u1ed3i n\u00e9n l\u1ea1i.\"\n        return SubmissionError(\"ZIP_ENTRIES\", msg, detail)\n    return SubmissionError(\n        \"ZIP_ENTRIES\",\n        f\"Zip c\u00f3 {count} m\u1ee5c nh\u01b0ng kh\u00f4ng c\u00f3 t\u1ec7p \u00ab{required}\u00bb. C\u00e1c m\u1ee5c t\u00ecm th\u1ea5y: {_list(names)}. \"\n        f\"Zip ph\u1ea3i ch\u1ee9a \u0111\u00fang m\u1ed9t t\u1ec7p {required}.\",\n        detail,\n    )\n\n\ndef path_traversal(names: list[str], why: str = \"\u0111\u01b0\u1eddng d\u1eabn tuy\u1ec7t \u0111\u1ed1i ho\u1eb7c ch\u1ee9a \u00ab..\u00bb\") -> SubmissionError:\n    return SubmissionError(\n        \"PATH_TRAVERSAL\",\n        f\"Zip ch\u1ee9a m\u1ee5c c\u00f3 \u0111\u01b0\u1eddng d\u1eabn kh\u00f4ng an to\u00e0n ({why}): {_list(names)}. \"\n        \"Zip ch\u1ec9 \u0111\u01b0\u1ee3c ch\u1ee9a t\u1ec7p model.onnx \u1edf g\u1ed1c, kh\u00f4ng c\u00f3 \u0111\u01b0\u1eddng d\u1eabn tuy\u1ec7t \u0111\u1ed1i, \u00ab..\u00bb hay li\u00ean k\u1ebft t\u01b0\u1ee3ng tr\u01b0ng (symlink).\",\n        {\"entries\": names, \"reason\": why},\n    )\n\n\n# ---------------------------------------------------------------------------------------------- ONNX validator\ndef onnx_parse(reason: str, hint: str = \"\") -> SubmissionError:\n    reason = reason.rstrip(\". \")\n    if hint:\n        msg = f\"Kh\u00f4ng \u0111\u1ecdc \u0111\u01b0\u1ee3c t\u1ec7p ONNX: {reason}. {hint}\"\n    else:\n        msg = (f\"Kh\u00f4ng \u0111\u1ecdc \u0111\u01b0\u1ee3c t\u1ec7p ONNX: {reason}. T\u1ec7p c\u00f3 th\u1ec3 b\u1ecb c\u1eaft c\u1ee5t ho\u1eb7c h\u1ecfng khi export/t\u1ea3i l\u00ean: \"\n               \"h\u00e3y export l\u1ea1i b\u1eb1ng \u00f4 export c\u1ee7a notebook v\u00e0 ki\u1ec3m tra \u00f4 t\u1ef1 ki\u1ec3m tra tr\u01b0\u1edbc khi n\u1ed9p.\")\n    return SubmissionError(\"ONNX_PARSE\", msg, {\"reason\": reason})\n\n\ndef onnx_external_data(tensor_name: str) -> SubmissionError:\n    return SubmissionError(\n        \"ONNX_EXTERNAL_DATA\",\n        f\"T\u1ec7p ONNX tham chi\u1ebfu d\u1eef li\u1ec7u tr\u1ecdng s\u1ed1 \u1edf t\u1ec7p ngo\u00e0i (tensor \u00ab{tensor_name}\u00bb). M\u1ecdi tr\u1ecdng s\u1ed1 ph\u1ea3i n\u1eb1m trong ch\u00ednh model.onnx: \"\n        \"h\u00e3y export l\u1ea1i b\u1eb1ng notebook (kh\u00f4ng d\u00f9ng t\u00f9y ch\u1ecdn save_as_external_data / m\u00f4 h\u00ecnh > 2 GB).\",\n        {\"tensor\": tensor_name},\n    )\n\n\ndef onnx_custom_op(items: list[str]) -> SubmissionError:\n    return SubmissionError(\n        \"ONNX_CUSTOM_OP\",\n        f\"M\u00f4 h\u00ecnh ch\u1ee9a to\u00e1n t\u1eed/domain kh\u00f4ng chu\u1ea9n: {_list(items)}. Ch\u1ec9 ch\u1ea5p nh\u1eadn to\u00e1n t\u1eed chu\u1ea9n c\u1ee7a domain ai.onnx \"\n        \"nh\u01b0 file export YOLOv8n c\u1ee7a Ultralytics; h\u00e3y export l\u1ea1i b\u1eb1ng notebook.\",\n        {\"items\": items},\n    )\n\n\ndef onnx_io(what: str, found: str, expected: str, hint: str = \"\") -> SubmissionError:\n    msg = f\"{what} c\u1ee7a m\u00f4 h\u00ecnh sai: t\u00ecm th\u1ea5y {found}, c\u1ea7n {expected}.\"\n    if hint:\n        msg += \" \" + hint\n    return SubmissionError(\"ONNX_IO\", msg, {\"what\": what, \"found\": found, \"expected\": expected})\n\n\ndef onnx_io_shape(what: str, found: Iterable[Any], expected: str, hint: str = \"\") -> SubmissionError:\n    return onnx_io(what, _shape(found), expected, hint)\n\n\ndef onnx_dtype(what: str, found: str, expected: str = \"float32 (fp32)\", hint: str = \"\") -> SubmissionError:\n    msg = f\"{what} d\u00f9ng ki\u1ec3u {found}, c\u1ea7n {expected}.\"\n    msg += \" \" + (hint or \"H\u00e3y export v\u1edbi half=False (kh\u00f4ng d\u00f9ng fp16/int8/bfloat16).\")\n    return SubmissionError(\"ONNX_DTYPE\", msg, {\"what\": what, \"found\": found, \"expected\": expected})\n\n\ndef structure_class_count(found_nc: int, expected_nc: int, names: Iterable[str]) -> SubmissionError:\n    msg = (f\"M\u00f4 h\u00ecnh c\u00f3 {found_nc} l\u1edbp (output c\u00f3 {4 + found_nc} k\u00eanh) nh\u01b0ng cu\u1ed9c thi c\u1ea7n \u0111\u00fang {expected_nc} l\u1edbp \"\n           f\"({', '.join(names)}, theo \u0111\u00fang th\u1ee9 t\u1ef1). \")\n    if found_nc == 80:\n        msg += \"80 l\u1edbp l\u00e0 c\u1ee7a YOLOv8n COCO g\u1ed1c: h\u00e3y fine-tune tr\u00ean d\u1eef li\u1ec7u c\u1ee7a nh\u00f3m v\u1edbi data.yaml c\u00f3 nc \u0111\u00fang r\u1ed3i export l\u1ea1i.\"\n    else:\n        msg += \"H\u00e3y ki\u1ec3m tra data.yaml (nc v\u00e0 names) d\u00f9ng khi hu\u1ea5n luy\u1ec7n, hu\u1ea5n luy\u1ec7n l\u1ea1i r\u1ed3i export l\u1ea1i.\"\n    return SubmissionError(\"STRUCTURE_MISMATCH\", msg, {\"reason\": \"class_count\", \"found_nc\": found_nc, \"expected_nc\": expected_nc})\n\n\ndef structure_mismatch(reason_vi: str, kind: str, **detail: Any) -> SubmissionError:\n    return SubmissionError(\n        \"STRUCTURE_MISMATCH\",\n        f\"Ki\u1ebfn tr\u00fac m\u00f4 h\u00ecnh kh\u00f4ng kh\u1edbp YOLOv8n: {reason_vi}\",\n        {\"reason\": kind, **detail},\n    )\n\n\ndef meta_names(expected: list[str], found: Any, why: str) -> SubmissionError:\n    return SubmissionError(\n        \"META_NAMES\",\n        f\"Metadata \u00abnames\u00bb c\u1ee7a m\u00f4 h\u00ecnh sai ({why}). C\u1ea7n \u0111\u00fang th\u1ee9 t\u1ef1: {', '.join(expected)}; \"\n        f\"t\u00ecm th\u1ea5y: {found}. H\u00e3y hu\u1ea5n luy\u1ec7n v\u1edbi data.yaml c\u00f3 names \u0111\u00fang th\u1ee9 t\u1ef1 r\u1ed3i export l\u1ea1i (kh\u00f4ng s\u1eeda tay metadata).\",\n        {\"expected\": expected, \"found\": found if isinstance(found, (list, dict, str, int, float, type(None))) else str(found)},\n    )\n\n\ndef pretrained_model() -> SubmissionError:\n    return SubmissionError(\n        \"PRETRAINED_MODEL\",\n        \"\u0110\u00e2y l\u00e0 model YOLOv8n pretrained g\u1ed1c c\u1ee7a Ultralytics (COCO). H\u00e3y fine-tune n\u00f3 tr\u00ean d\u1eef li\u1ec7u c\u1ee7a nh\u00f3m r\u1ed3i export l\u1ea1i; \"\n        \"n\u1ed9p l\u1ea1i m\u00f4 h\u00ecnh g\u1ed1c kh\u00f4ng \u0111\u01b0\u1ee3c ch\u1ea5p nh\u1eadn v\u00e0 kh\u00f4ng b\u1ecb tr\u1eeb l\u01b0\u1ee3t n\u1ed9p.\",\n        {},\n    )\n\n\ndef baseline_model() -> SubmissionError:\n    return SubmissionError(\n        \"BASELINE_MODEL\",\n        \"\u0110\u00e2y l\u00e0 m\u00f4 h\u00ecnh baseline do ban t\u1ed5 ch\u1ee9c cung c\u1ea5p, kh\u00f4ng ph\u1ea3i k\u1ebft qu\u1ea3 hu\u1ea5n luy\u1ec7n c\u1ee7a nh\u00f3m. \"\n        \"H\u00e3y hu\u1ea5n luy\u1ec7n c\u1ea3i ti\u1ebfn r\u1ed3i n\u1ed9p m\u00f4 h\u00ecnh c\u1ee7a nh\u00f3m (kh\u00f4ng b\u1ecb tr\u1eeb l\u01b0\u1ee3t n\u1ed9p).\",\n        {},\n    )\n\n\ndef invalid_weights(kind: str, **detail: Any) -> SubmissionError:\n    \"\"\"``kind``: ``nonfinite`` (NaN/Inf in a Conv weight or bias) or ``all_zero`` (every Conv kernel is exactly 0).\"\"\"\n    if kind == \"all_zero\":\n        message = (\n            \"M\u1ecdi tr\u1ecdng s\u1ed1 Conv c\u1ee7a m\u00f4 h\u00ecnh \u0111\u1ec1u b\u1eb1ng 0 n\u00ean \u0111\u00e2y kh\u00f4ng ph\u1ea3i m\u00f4 h\u00ecnh \u0111\u00e3 hu\u1ea5n luy\u1ec7n. \"\n            \"H\u00e3y ki\u1ec3m tra BEST_WEIGHTS tr\u1ecf \u0111\u00fang t\u1ec7p best.pt/last.pt c\u1ee7a l\u1ea7n hu\u1ea5n luy\u1ec7n, r\u1ed3i export l\u1ea1i b\u1eb1ng notebook \"\n            \"(b\u00e0i n\u1ed9p kh\u00f4ng b\u1ecb tr\u1eeb l\u01b0\u1ee3t).\"\n        )\n    else:\n        n, total, tensors = detail.get(\"bad_elements\", 0), detail.get(\"total_elements\", 0), detail.get(\"bad_tensors\", 0)\n        message = (\n            f\"Tr\u1ecdng s\u1ed1 m\u00f4 h\u00ecnh ch\u1ee9a gi\u00e1 tr\u1ecb NaN ho\u1eb7c Inf ({n:,}/{total:,} ph\u1ea7n t\u1eed, {tensors} tensor Conv). \"\n            \"Th\u01b0\u1eddng do hu\u1ea5n luy\u1ec7n b\u1ecb ph\u00e2n k\u1ef3 (loss = nan). H\u00e3y gi\u1ea3m lr0, gi\u1ea3m augmentation ho\u1eb7c \u0111\u1eb7t amp=False, \"\n            \"d\u00f9ng l\u1ea1i t\u1ec7p best.pt tr\u01b0\u1edbc khi ph\u00e2n k\u1ef3 r\u1ed3i export l\u1ea1i b\u1eb1ng notebook (b\u00e0i n\u1ed9p kh\u00f4ng b\u1ecb tr\u1eeb l\u01b0\u1ee3t).\"\n        )\n    return SubmissionError(\"INVALID_WEIGHTS\", message, {\"kind\": kind, **detail})\n\n\n# ---------------------------------------------------------------------------------------------- container run\ndef time_limit(seconds: float, stage: str = \"ch\u1ea1y m\u00f4 h\u00ecnh\") -> SubmissionError:\n    return SubmissionError(\n        \"TIME_LIMIT\",\n        f\"Qu\u00e1 th\u1eddi gian cho ph\u00e9p khi {stage} (gi\u1edbi h\u1ea1n {seconds:g} gi\u00e2y). B\u00e0i n\u1ed9p kh\u00f4ng b\u1ecb tr\u1eeb l\u01b0\u1ee3t: \"\n        \"h\u00e3y ki\u1ec3m tra m\u00f4 h\u00ecnh l\u00e0 YOLOv8n chu\u1ea9n, kh\u00f4ng ch\u1ee9a v\u00f2ng l\u1eb7p/ti\u1ec1n x\u1eed l\u00fd n\u1eb7ng, r\u1ed3i n\u1ed9p l\u1ea1i.\",\n        {\"limit_s\": seconds},\n    )\n\n\ndef memory_limit(mb: float) -> SubmissionError:\n    return SubmissionError(\n        \"MEMORY_LIMIT\",\n        f\"M\u00f4 h\u00ecnh v\u01b0\u1ee3t gi\u1edbi h\u1ea1n b\u1ed9 nh\u1edb {mb:g} MB khi ch\u1ea1y. B\u00e0i n\u1ed9p kh\u00f4ng b\u1ecb tr\u1eeb l\u01b0\u1ee3t: h\u00e3y d\u00f9ng YOLOv8n chu\u1ea9n (fp32, imgsz 640) r\u1ed3i n\u1ed9p l\u1ea1i.\",\n        {\"limit_mb\": mb},\n    )\n\n\ndef runtime_error(reason: str) -> SubmissionError:\n    return SubmissionError(\n        \"RUNTIME_ERROR\",\n        f\"H\u1ec7 th\u1ed1ng kh\u00f4ng ch\u1ea1y \u0111\u01b0\u1ee3c m\u00f4 h\u00ecnh ({reason}). B\u00e0i n\u1ed9p kh\u00f4ng b\u1ecb tr\u1eeb l\u01b0\u1ee3t: h\u00e3y ch\u1ea1y \u00f4 t\u1ef1 ki\u1ec3m tra trong notebook; \"\n        \"n\u1ebfu v\u1eabn l\u1ed7i, g\u1eedi th\u00f4ng b\u00e1o n\u00e0y cho ban t\u1ed5 ch\u1ee9c.\",\n        {\"reason\": reason},\n    )\n",
  "fingerprint.py": "\"\"\"Tensor fingerprints and file hashes.\n\n* ``tensor_fingerprint``: SHA-256 over ALL initializers sorted by name, each as\n  ``name \\\\0 dtype \\\\0 repr(shape) \\\\0 raw little-endian bytes`` (design spec section 6). Two exports of the\n  same checkpoint differ in file bytes (Ultralytics writes the export date into the metadata) but have the\n  same tensor fingerprint.\n* ``content_fingerprint``: a name- and layout-independent copy detector over the Conv weights/biases only\n  (see ``signature.conv_tensors``). It stays equal across simplify/opset/renaming variants of the same\n  weights, so it also catches a copier who re-exports with other settings.\n* ``file_sha256``: audit only.\n\nTensor bytes are decoded defensively: dims are bounded before any array is built, raw_data length must\nmatch the declared shape, and external-data references are refused by the caller before this module runs.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport numpy as np\nfrom onnx import TensorProto\n\nfrom . import errors\n\n# ONNX element type -> numpy dtype name (as in str(np.dtype)) and item size in bytes (None = unknown/variable).\n_DTYPE_NAMES: dict[int, tuple[str, int | None]] = {\n    TensorProto.FLOAT: (\"float32\", 4),\n    TensorProto.DOUBLE: (\"float64\", 8),\n    TensorProto.FLOAT16: (\"float16\", 2),\n    TensorProto.BFLOAT16: (\"bfloat16\", 2),\n    TensorProto.INT8: (\"int8\", 1),\n    TensorProto.UINT8: (\"uint8\", 1),\n    TensorProto.INT16: (\"int16\", 2),\n    TensorProto.UINT16: (\"uint16\", 2),\n    TensorProto.INT32: (\"int32\", 4),\n    TensorProto.UINT32: (\"uint32\", 4),\n    TensorProto.INT64: (\"int64\", 8),\n    TensorProto.UINT64: (\"uint64\", 8),\n    TensorProto.BOOL: (\"bool\", 1),\n    TensorProto.COMPLEX64: (\"complex64\", 8),\n    TensorProto.COMPLEX128: (\"complex128\", 16),\n}\n\n\ndef dtype_name(elem_type: int) -> str:\n    known = _DTYPE_NAMES.get(elem_type)\n    if known:\n        return known[0]\n    try:\n        return TensorProto.DataType.Name(elem_type).lower()\n    except ValueError:\n        return f\"onnx_dtype_{elem_type}\"\n\n\ndef is_non_fp32_float(elem_type: int) -> bool:\n    \"\"\"True for any floating-point element type other than float32 (fp16, bf16, fp64, fp8, fp4...).\"\"\"\n    if elem_type == TensorProto.FLOAT:\n        return False\n    try:\n        name = TensorProto.DataType.Name(elem_type)\n    except ValueError:\n        return False\n    return name.startswith((\"FLOAT\", \"BFLOAT\", \"DOUBLE\"))\n\n\ndef sha256_file(path: str | Path) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b\"\"):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef tensor_shape(t: TensorProto, max_elems: int | None = None) -> tuple[int, ...]:\n    \"\"\"Declared dims, validated (non-negative, bounded element count).\"\"\"\n    n = 1\n    for d in t.dims:\n        if d < 0:\n            raise errors.onnx_parse(f\"tensor \u00ab{t.name}\u00bb c\u00f3 k\u00edch th\u01b0\u1edbc \u00e2m\")\n        n *= d\n        if max_elems is not None and n > max_elems:\n            raise errors.structure_mismatch(\n                f\"tensor \u00ab{t.name}\u00bb c\u00f3 qu\u00e1 nhi\u1ec1u ph\u1ea7n t\u1eed (khai b\u00e1o {n:,}+), v\u01b0\u1ee3t gi\u1edbi h\u1ea1n {max_elems:,}.\", \"tensor_too_large\",\n                tensor=t.name,\n            )\n    return tuple(int(d) for d in t.dims)\n\n\ndef tensor_bytes(t: TensorProto, max_elems: int | None = None) -> tuple[str, tuple[int, ...], bytes]:\n    \"\"\"(dtype name, shape, canonical little-endian C-order bytes) of one TensorProto.\"\"\"\n    shape = tensor_shape(t, max_elems)\n    numel = 1\n    for d in shape:\n        numel *= d\n    dt = dtype_name(t.data_type)\n    item = _DTYPE_NAMES.get(t.data_type, (None, None))[1]\n\n    if t.data_type == TensorProto.STRING:\n        return dt, shape, b\"\\x00\".join(bytes(s) for s in t.string_data)\n    if t.HasField(\"raw_data\"):\n        raw = bytes(t.raw_data)\n        if item is not None and len(raw) != numel * item:\n            raise errors.onnx_parse(\n                f\"tensor \u00ab{t.name}\u00bb khai b\u00e1o {numel} ph\u1ea7n t\u1eed {dt} nh\u01b0ng ch\u1ee9a {len(raw)} byte (c\u1ea7n {numel * item})\"\n            )\n        return dt, shape, raw\n    if item is None:\n        # unknown packed type stored in typed fields: hash what is there deterministically\n        return dt, shape, repr((list(t.int32_data), list(t.int64_data))).encode()\n    from onnx import numpy_helper\n\n    try:\n        arr = numpy_helper.to_array(t)\n    except Exception as exc:  # noqa: BLE001 - any decode failure is a corrupt tensor\n        raise errors.onnx_parse(f\"kh\u00f4ng gi\u1ea3i m\u00e3 \u0111\u01b0\u1ee3c tensor \u00ab{t.name}\u00bb: {str(exc)[:100]}\") from exc\n    arr = np.ascontiguousarray(arr)\n    if arr.dtype.byteorder == \">\":\n        arr = arr.astype(arr.dtype.newbyteorder(\"<\"))\n    return dt, shape, arr.tobytes()\n\n\ndef tensor_fingerprint(initializers: Iterable[TensorProto], max_elems: int | None = None) -> str:\n    \"\"\"SHA-256 over all initializers sorted by name as (name, dtype, shape, raw bytes).\"\"\"\n    h = hashlib.sha256()\n    for t in sorted(initializers, key=lambda x: x.name):\n        dt, shape, raw = tensor_bytes(t, max_elems)\n        shape = shape or (1,)  # scalars are hashed as shape (1,): the convention of the reference fixtures (np.ascontiguousarray)\n        h.update(t.name.encode() + b\"\\0\" + dt.encode() + b\"\\0\" + repr(shape).encode() + b\"\\0\")\n        h.update(raw)\n    return h.hexdigest()\n\n\ndef model_tensor_fingerprint(model, max_elems: int | None = None) -> str:\n    return tensor_fingerprint(model.graph.initializer, max_elems)\n",
  "package.py": "\"\"\"Cheap, safe checks on a submission zip. Nothing here parses ONNX.\n\n``inspect_zip`` is what the upload endpoint calls: it reads only the zip central directory and enforces\nthe limits (size, entry layout, path safety, compression ratio). ``extract_model`` streams the single\n``model.onnx`` entry to disk with a hard byte cap, so a lying header cannot blow up memory or disk.\n\"\"\"\nfrom __future__ import annotations\n\nimport io\nimport os\nimport stat\nimport struct\nimport zipfile\nfrom dataclasses import dataclass\nfrom pathlib import Path, PurePosixPath\nfrom typing import BinaryIO, Union\n\nfrom . import errors\nfrom .config import Config, load_config\n\nSource = Union[str, \"os.PathLike[str]\", bytes, bytearray, BinaryIO]\n\n_EOCD_SIG = b\"PK\\x05\\x06\"\n_EOCD_MIN = 22\n_MAX_COMMENT = 0xFFFF\n_CHUNK = 1 << 20\n_ALLOWED_COMPRESSION = (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED)\n\n\n@dataclass(frozen=True)\nclass ZipSummary:\n    \"\"\"Result of a successful ``inspect_zip``.\"\"\"\n\n    entry_name: str\n    zip_bytes: int\n    compressed_bytes: int\n    uncompressed_bytes: int\n\n    @property\n    def ratio(self) -> float:\n        return self.uncompressed_bytes / max(self.compressed_bytes, 1)\n\n    def to_dict(self) -> dict:\n        return {\n            \"entry_name\": self.entry_name,\n            \"zip_bytes\": self.zip_bytes,\n            \"compressed_bytes\": self.compressed_bytes,\n            \"uncompressed_bytes\": self.uncompressed_bytes,\n            \"ratio\": round(self.ratio, 3),\n        }\n\n\n# ------------------------------------------------------------------------------------------ upload layer\ndef check_extension(filename: str, required: str = \".zip\") -> None:\n    \"\"\"Upload layer only: the original file name must end in ``.zip`` (case-insensitive).\"\"\"\n    if not isinstance(filename, str) or not filename.lower().endswith(required):\n        raise errors.bad_extension(filename if isinstance(filename, str) else \"\", required)\n\n\n# ------------------------------------------------------------------------------------------ helpers\nclass _Opened:\n    \"\"\"Context manager yielding (seekable binary file, total size) for any accepted ``Source``.\"\"\"\n\n    def __init__(self, source: Source):\n        self._source = source\n        self._owned: BinaryIO | None = None\n        self.fileobj: BinaryIO | None = None\n        self.size = 0\n\n    def __enter__(self) -> \"_Opened\":\n        src = self._source\n        if isinstance(src, (bytes, bytearray)):\n            self.fileobj = io.BytesIO(bytes(src))\n        elif isinstance(src, (str, os.PathLike)):\n            try:\n                self._owned = open(src, \"rb\")\n            except OSError as exc:\n                raise errors.zip_corrupt(f\"kh\u00f4ng m\u1edf \u0111\u01b0\u1ee3c t\u1ec7p: {exc.strerror or exc}\") from exc\n            self.fileobj = self._owned\n        else:\n            self.fileobj = src  # already a binary file object\n        self.fileobj.seek(0, os.SEEK_END)\n        self.size = self.fileobj.tell()\n        self.fileobj.seek(0)\n        return self\n\n    def __exit__(self, *exc) -> None:\n        if self._owned is not None:\n            self._owned.close()\n\n\ndef _read_eocd(f: BinaryIO, size: int) -> tuple[int, int]:\n    \"\"\"Return (entry_count, central_directory_size) from the End Of Central Directory record.\"\"\"\n    if size < _EOCD_MIN:\n        raise errors.zip_corrupt(\"t\u1ec7p qu\u00e1 nh\u1ecf \u0111\u1ec3 l\u00e0 m\u1ed9t t\u1ec7p zip\")\n    tail_len = min(size, _EOCD_MIN + _MAX_COMMENT)\n    f.seek(size - tail_len)\n    tail = f.read(tail_len)\n    pos = tail.rfind(_EOCD_SIG)\n    if pos < 0 or pos + _EOCD_MIN > len(tail):\n        raise errors.zip_corrupt(\"kh\u00f4ng t\u00ecm th\u1ea5y central directory c\u1ee7a zip, t\u1ec7p c\u00f3 th\u1ec3 b\u1ecb c\u1eaft c\u1ee5t\")\n    _sig, _disk, _cd_disk, _n_disk, n_total, cd_size, _cd_off, _clen = struct.unpack(\"<4sHHHHIIH\", tail[pos:pos + _EOCD_MIN])\n    return n_total, cd_size\n\n\ndef _unsafe_name(name: str) -> str | None:\n    \"\"\"Why ``name`` is unsafe as an entry path, or None.\"\"\"\n    if not name:\n        return \"t\u00ean r\u1ed7ng\"\n    if \"\\x00\" in name or any(ord(c) < 32 for c in name):\n        return \"ch\u1ee9a k\u00fd t\u1ef1 \u0111i\u1ec1u khi\u1ec3n\"\n    norm = name.replace(\"\\\\\", \"/\")\n    if norm.startswith(\"/\"):\n        return \"\u0111\u01b0\u1eddng d\u1eabn tuy\u1ec7t \u0111\u1ed1i\"\n    if len(norm) >= 2 and norm[1] == \":\" and norm[0].isalpha():\n        return \"\u0111\u01b0\u1eddng d\u1eabn \u1ed5 \u0111\u0129a Windows\"\n    if \"..\" in PurePosixPath(norm).parts:\n        return \"ch\u1ee9a \u00ab..\u00bb\"\n    return None\n\n\ndef _entry_is_symlink(info: zipfile.ZipInfo) -> bool:\n    if info.create_system != 3:  # unix\n        return False\n    return stat.S_ISLNK(info.external_attr >> 16)\n\n\ndef _open_zip(f: BinaryIO, cfg: Config) -> zipfile.ZipFile:\n    try:\n        return zipfile.ZipFile(f)\n    except (zipfile.BadZipFile, EOFError, ValueError, NotImplementedError, OSError, struct.error) as exc:\n        raise errors.zip_corrupt(str(exc)[:120] or type(exc).__name__) from exc\n\n\n# ------------------------------------------------------------------------------------------ inspect\ndef _inspect(zf: zipfile.ZipFile, size: int, cfg: Config) -> tuple[ZipSummary, zipfile.ZipInfo]:\n    infos = zf.infolist()\n    names = [i.filename for i in infos]\n    if not infos:\n        raise errors.zip_entries_empty()\n\n    bad = [(n, _unsafe_name(n)) for n in names]\n    bad = [(n, why) for n, why in bad if why]\n    if bad:\n        raise errors.path_traversal([n for n, _ in bad], bad[0][1])\n    links = [i.filename for i in infos if _entry_is_symlink(i)]\n    if links:\n        raise errors.path_traversal(links, \"li\u00ean k\u1ebft t\u01b0\u1ee3ng tr\u01b0ng (symlink)\")\n\n    required = cfg.required_entry\n    if names != [required]:\n        raise errors.zip_entries_wrong(names, required)\n\n    info = infos[0]\n    if info.flag_bits & 0x1:\n        raise errors.zip_corrupt(\"zip c\u00f3 \u0111\u1eb7t m\u1eadt kh\u1ea9u/m\u00e3 h\u00f3a, h\u00e3y t\u1ea1o zip kh\u00f4ng m\u1eadt kh\u1ea9u\")\n    if info.compress_type not in _ALLOWED_COMPRESSION:\n        raise errors.zip_corrupt(f\"ki\u1ec3u n\u00e9n {info.compress_type} kh\u00f4ng \u0111\u01b0\u1ee3c h\u1ed7 tr\u1ee3, h\u00e3y d\u00f9ng n\u00e9n zip th\u00f4ng th\u01b0\u1eddng (deflate)\")\n\n    total_uncompressed = sum(i.file_size for i in infos)\n    if info.file_size > cfg.max_uncompressed_bytes or total_uncompressed > cfg.max_uncompressed_bytes:\n        raise errors.zip_bomb(\n            f\"{required} khai b\u00e1o {info.file_size / 1048576:.1f} MB sau gi\u1ea3i n\u00e9n, gi\u1edbi h\u1ea1n {cfg.max_uncompressed_mb:.0f} MB\",\n            declared_bytes=info.file_size, limit_bytes=cfg.max_uncompressed_bytes,\n        )\n    if info.file_size > cfg.ratio_check_min_bytes:\n        ratio = info.file_size / max(info.compress_size, 1)\n        if ratio > cfg.max_compression_ratio:\n            raise errors.zip_bomb(\n                f\"t\u1ec9 l\u1ec7 n\u00e9n {ratio:.0f}:1, v\u01b0\u1ee3t ng\u01b0\u1ee1ng {cfg.max_compression_ratio:.0f}:1\",\n                ratio=round(ratio, 1), limit_ratio=cfg.max_compression_ratio,\n            )\n    summary = ZipSummary(\n        entry_name=info.filename, zip_bytes=size,\n        compressed_bytes=info.compress_size, uncompressed_bytes=info.file_size,\n    )\n    return summary, info\n\n\ndef inspect_zip(source: Source, config: Config | None = None) -> ZipSummary:\n    \"\"\"Cheap safety checks. Raises ``SubmissionError``; never parses ONNX.\n\n    Order: total size, central-directory sanity (entry count / size read from the EOCD before any\n    entry is materialised), path safety, exactly one entry named ``model.onnx``, encryption and\n    compression kind, declared uncompressed size and compression ratio.\n    \"\"\"\n    cfg = config or load_config()\n    with _Opened(source) as op:\n        assert op.fileobj is not None\n        if op.size > cfg.max_zip_bytes:\n            raise errors.zip_too_large(op.size, cfg.max_zip_bytes)\n        n_total, cd_size = _read_eocd(op.fileobj, op.size)\n        if n_total > cfg.max_zip_entries_scanned or cd_size > cfg.max_central_directory_bytes:\n            raise errors.zip_entries_too_many(n_total, cfg.max_zip_entries_scanned)\n        op.fileobj.seek(0)\n        with _open_zip(op.fileobj, cfg) as zf:\n            summary, _info = _inspect(zf, op.size, cfg)\n    return summary\n\n\n# ------------------------------------------------------------------------------------------ extract\ndef extract_model(source: Source, dest_dir: str | os.PathLike, config: Config | None = None,\n                  summary: ZipSummary | None = None) -> Path:\n    \"\"\"Stream the single ``model.onnx`` entry to ``dest_dir/model.onnx`` and return that path.\n\n    The destination file name is fixed (the entry name is never used to build a path). Writing stops with\n    ``ZIP_BOMB`` as soon as the real decompressed size passes the cap, whatever the headers claim.\n    \"\"\"\n    cfg = config or load_config()\n    dest = Path(dest_dir)\n    dest.mkdir(parents=True, exist_ok=True)\n    out_path = dest / \"model.onnx\"\n    with _Opened(source) as op:\n        assert op.fileobj is not None\n        op.fileobj.seek(0)\n        if summary is None:\n            inspect_zip(_rewind(op.fileobj), cfg)\n            op.fileobj.seek(0)\n        with _open_zip(op.fileobj, cfg) as zf:\n            _summary, info = _inspect(zf, op.size, cfg)\n            cap = cfg.max_uncompressed_bytes\n            written = 0\n            try:\n                with zf.open(info) as src, open(out_path, \"xb\") as dst:\n                    while True:\n                        chunk = src.read(_CHUNK)\n                        if not chunk:\n                            break\n                        written += len(chunk)\n                        if written > cap:\n                            raise errors.zip_bomb(\n                                f\"d\u1eef li\u1ec7u th\u1ef1c t\u1ebf v\u01b0\u1ee3t {cfg.max_uncompressed_mb:.0f} MB khi gi\u1ea3i n\u00e9n\",\n                                limit_bytes=cap,\n                            )\n                        dst.write(chunk)\n            except errors.SubmissionError:\n                out_path.unlink(missing_ok=True)\n                raise\n            except FileExistsError:\n                raise\n            except (zipfile.BadZipFile, EOFError, OSError, ValueError, NotImplementedError, RuntimeError) as exc:\n                out_path.unlink(missing_ok=True)\n                raise errors.zip_corrupt(f\"l\u1ed7i khi gi\u1ea3i n\u00e9n: {str(exc)[:100]}\") from exc\n            if written != info.file_size:\n                out_path.unlink(missing_ok=True)\n                raise errors.zip_corrupt(\"k\u00edch th\u01b0\u1edbc gi\u1ea3i n\u00e9n kh\u00f4ng kh\u1edbp v\u1edbi khai b\u00e1o trong zip\")\n    return out_path\n\n\ndef _rewind(f: BinaryIO) -> BinaryIO:\n    f.seek(0)\n    return f\n",
  "signature.py": "\"\"\"Tolerant weight signature of a YOLOv8n export (instructor clarification 6).\n\nColab's torch/onnx/onnxslim versions differ from the machine that built the reference, so comparing every\nconstant or op would reject honest students. The shipped reference keeps only what is stable:\n\n* the multiset of ``(role, dtype, shape)`` of every Conv weight (role 0) and bias (role 1), resolved\n  through ``Identity``/``Constant`` nodes (raw ``simplify=False`` traces put an Identity between each\n  initializer and its Conv), without names or values;\n* the number of Conv nodes;\n* the set of allowed op types (union over the contract export and its simplify/opset variants).\n\n``signature_sha256`` is ``sha256(repr([(role, dtype, shape), ...]))`` over the sorted tuples, identical to the\n``conv_signature_sha256`` that ``scripts/make_fixtures.sh --describe`` prints.\n\n``content_fingerprint`` additionally hashes the Conv values (still name-independent) and is the stronger\ncopy detector used next to the plain tensor fingerprint.\n\"\"\"\nfrom __future__ import annotations\n\nimport collections\nimport hashlib\nimport json\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nfrom onnx import AttributeProto, TensorProto\n\nfrom . import errors\nfrom .fingerprint import sha256_bytes, tensor_bytes\n\nSCHEMA_VERSION = 1\n\nTriple = tuple  # (role: int, dtype: str, shape: tuple[int, ...])\n\n\n# ------------------------------------------------------------------------------------------ tensor lookup\ndef constant_tensors(model) -> dict[str, TensorProto]:\n    \"\"\"Output name -> TensorProto for ``Constant`` nodes with a ``value`` tensor attribute.\"\"\"\n    out: dict[str, TensorProto] = {}\n    for n in model.graph.node:\n        if n.op_type != \"Constant\" or len(n.output) != 1:\n            continue\n        for a in n.attribute:\n            if a.name == \"value\" and a.type == AttributeProto.TENSOR:\n                out[n.output[0]] = a.t\n    return out\n\n\ndef iter_conv_params(model):\n    \"\"\"(role, TensorProto or None) of every Conv weight (role 0) and bias (role 1); None = not a constant in the file.\"\"\"\n    inits = {t.name: t for t in model.graph.initializer}\n    consts = constant_tensors(model)\n    producer = {o: n for n in model.graph.node for o in n.output}\n\n    def resolve(name: str):\n        for _ in range(64):  # bounded Identity chain\n            if name in inits:\n                return inits[name]\n            if name in consts:\n                return consts[name]\n            node = producer.get(name)\n            if node is not None and node.op_type == \"Identity\" and node.input:\n                name = node.input[0]\n                continue\n            return None\n        return None\n\n    for n in model.graph.node:\n        if n.op_type != \"Conv\":\n            continue\n        for role, name in enumerate(n.input[1:3]):\n            if name == \"\":\n                continue  # optional bias not provided\n            yield role, resolve(name)\n\n\ndef conv_tensors(model, max_elems: int | None = None) -> list[tuple]:\n    \"\"\"(role, dtype, shape, sha256(bytes)) of every Conv weight (role 0) and bias (role 1), sorted.\"\"\"\n    out: list[tuple] = []\n    for role, t in iter_conv_params(model):\n        if t is None:\n            out.append((role, \"unresolved\", (), \"\"))\n        else:\n            dt, shape, raw = tensor_bytes(t, max_elems)\n            out.append((role, dt, shape, sha256_bytes(raw)))\n    return sorted(out)\n\n\ndef signature_digest(triples: Iterable[Triple]) -> str:\n    return sha256_bytes(repr([(r, d, s) for r, d, s in triples]).encode())\n\n\ndef content_fingerprint(tensors: list[tuple]) -> str:\n    \"\"\"Name-independent hash of Conv (role, dtype, shape, value hash): the stronger duplicate detector.\"\"\"\n    return sha256_bytes(repr(tensors).encode())\n\n\n@dataclass\nclass WeightSignature:\n    triples: list[Triple]  # sorted (role, dtype, shape)\n    conv_nodes: int\n    op_types: dict[str, int]\n    digest: str\n    content_fingerprint: str\n    conv_params: int = 0\n    unresolved: int = 0\n\n    def counter(self) -> collections.Counter:\n        return collections.Counter(self.triples)\n\n    def summary(self) -> dict[str, Any]:\n        return {\n            \"signature_sha256\": self.digest, \"conv_nodes\": self.conv_nodes, \"conv_params\": self.conv_params,\n            \"content_fingerprint\": self.content_fingerprint, \"op_types\": self.op_types,\n        }\n\n\ndef compute_signature(model, max_elems: int | None = None) -> WeightSignature:\n    tensors = conv_tensors(model, max_elems)\n    triples = [(r, d, s) for r, d, s, _ in tensors]\n    ops = collections.Counter(n.op_type for n in model.graph.node)\n    params = 0\n    for r, d, s in triples:\n        if r == 0 and d != \"unresolved\":\n            p = 1\n            for x in s:\n                p *= x\n            params += p\n    return WeightSignature(\n        triples=triples,\n        conv_nodes=ops.get(\"Conv\", 0),\n        op_types=dict(sorted(ops.items())),\n        digest=signature_digest(triples),\n        content_fingerprint=content_fingerprint(tensors),\n        conv_params=params,\n        unresolved=sum(1 for t in triples if t[1] == \"unresolved\"),\n    )\n\n\n# ------------------------------------------------------------------------------------------ IO helpers\ndef tensor_type_dims(value_info) -> tuple[int, list[Any]]:\n    \"\"\"(elem_type, dims) of a ValueInfoProto; dims are ints for fixed sizes, else the symbolic name or None.\"\"\"\n    tt = value_info.type.tensor_type\n    dims: list[Any] = []\n    for d in tt.shape.dim:\n        which = d.WhichOneof(\"value\")\n        if which == \"dim_value\":\n            dims.append(int(d.dim_value))\n        elif which == \"dim_param\":\n            dims.append(d.dim_param)\n        else:\n            dims.append(None)\n    return int(tt.elem_type), dims\n\n\ndef graph_inputs(model) -> list:\n    \"\"\"Real graph inputs (older IR versions also list initializers as inputs).\"\"\"\n    init = {t.name for t in model.graph.initializer}\n    return [v for v in model.graph.input if v.name not in init]\n\n\ndef derive_nc(output_dims: list[Any]) -> int | None:\n    \"\"\"nc from an output of shape [1, 4+nc, anchors]; None if the channel dim is not a fixed int > 4.\"\"\"\n    if len(output_dims) == 3 and isinstance(output_dims[1], int) and output_dims[1] > 4:\n        return output_dims[1] - 4\n    return None\n\n\ndef derive_imgsz(input_dims: list[Any]) -> int | None:\n    \"\"\"Square image size from an input [1,3,H,W] when H == W are fixed ints.\"\"\"\n    if len(input_dims) == 4 and isinstance(input_dims[2], int) and input_dims[2] == input_dims[3]:\n        return input_dims[2]\n    return None\n\n\n# ------------------------------------------------------------------------------------------ reference\n@dataclass\nclass Reference:\n    nc: int\n    imgsz: int\n    triples: list[Triple]\n    signature_sha256: str\n    conv_nodes: int\n    conv_params: int\n    allowed_op_types: frozenset\n    op_types: dict[str, int] = field(default_factory=dict)\n    provenance: dict[str, Any] = field(default_factory=dict)\n    raw: dict[str, Any] = field(default_factory=dict, repr=False)\n\n    def counter(self) -> collections.Counter:\n        return collections.Counter(self.triples)\n\n    @classmethod\n    def from_json(cls, data: dict[str, Any]) -> \"Reference\":\n        if data.get(\"schema_version\") != SCHEMA_VERSION:\n            raise ValueError(f\"unsupported reference schema_version {data.get('schema_version')!r}\")\n        triples: list[Triple] = []\n        for row in data[\"conv_tensors\"]:\n            triples.extend([(int(row[\"role\"]), str(row[\"dtype\"]), tuple(int(x) for x in row[\"shape\"]))] * int(row[\"count\"]))\n        triples.sort()\n        digest = signature_digest(triples)\n        if digest != data[\"signature_sha256\"]:\n            raise ValueError(\"reference_signature.json is inconsistent: signature_sha256 does not match conv_tensors\")\n        return cls(\n            nc=int(data[\"nc\"]), imgsz=int(data[\"imgsz\"]), triples=triples, signature_sha256=digest,\n            conv_nodes=int(data[\"conv_nodes\"]), conv_params=int(data[\"conv_params\"]),\n            allowed_op_types=frozenset(data[\"allowed_op_types\"]), op_types=dict(data.get(\"reference_op_counts\", {})),\n            provenance=dict(data.get(\"provenance\", {})), raw=data,\n        )\n\n    @classmethod\n    def load(cls, path: str | Path) -> \"Reference\":\n        return cls.from_json(json.loads(Path(path).read_text(encoding=\"utf-8\")))\n\n\ndef build_reference_json(sig: WeightSignature, *, nc: int, imgsz: int, allowed_op_types: Iterable[str],\n                         provenance: dict[str, Any]) -> dict[str, Any]:\n    rows = [{\"role\": r, \"dtype\": d, \"shape\": list(s), \"count\": c} for (r, d, s), c in sorted(sig.counter().items())]\n    return {\n        \"schema_version\": SCHEMA_VERSION,\n        \"nc\": nc,\n        \"imgsz\": imgsz,\n        \"signature_sha256\": sig.digest,\n        \"conv_nodes\": sig.conv_nodes,\n        \"conv_params\": sig.conv_params,\n        \"conv_tensors\": rows,\n        \"allowed_op_types\": sorted(set(allowed_op_types)),\n        \"reference_op_counts\": sig.op_types,\n        \"provenance\": provenance,\n    }\n\n\n# ------------------------------------------------------------------------------------------ comparison\ndef _fmt_shape(s: tuple) -> str:\n    return \"x\".join(str(x) for x in s) if s else \"scalar\"\n\n\ndef diff_signature(ref: Reference, sig: WeightSignature) -> dict[str, Any]:\n    \"\"\"Machine-readable difference between a reference and a submission signature (empty lists = equal).\"\"\"\n    rc, sc = ref.counter(), sig.counter()\n    missing = sorted((rc - sc).elements())\n    extra = sorted((sc - rc).elements())\n    return {\n        \"equal\": not missing and not extra and ref.conv_nodes == sig.conv_nodes,\n        \"conv_nodes\": [ref.conv_nodes, sig.conv_nodes],\n        \"conv_params\": [ref.conv_params, sig.conv_params],\n        \"missing_tensors\": missing,   # in the reference but not in the submission\n        \"extra_tensors\": extra,       # in the submission but not in the reference\n        \"extra_op_types\": sorted(set(sig.op_types) - set(ref.allowed_op_types)),\n    }\n\n\ndef stem_channels(triples: Iterable[Triple]) -> list[int]:\n    \"\"\"Output channels of Conv weights that read the 3-channel image (the stem), sorted.\"\"\"\n    return sorted(s[0] for r, d, s in triples if r == 0 and len(s) == 4 and s[1] == 3)\n\n\ndef diagnose_mismatch(ref: Reference, sig: WeightSignature) -> errors.SubmissionError:\n    \"\"\"Turn a signature difference into a precise Vietnamese STRUCTURE_MISMATCH error.\"\"\"\n    d = diff_signature(ref, sig)\n    detail = {\"conv_nodes\": d[\"conv_nodes\"], \"conv_params\": d[\"conv_params\"],\n              \"missing_tensors\": [[r, dt, list(s)] for r, dt, s in d[\"missing_tensors\"][:10]],\n              \"extra_tensors\": [[r, dt, list(s)] for r, dt, s in d[\"extra_tensors\"][:10]]}\n    if sig.unresolved:\n        return errors.structure_mismatch(\n            f\"{sig.unresolved} tham s\u1ed1 Conv kh\u00f4ng ph\u1ea3i l\u00e0 tr\u1ecdng s\u1ed1 c\u1ed1 \u0111\u1ecbnh trong t\u1ec7p (\u0111\u01b0\u1ee3c t\u00ednh to\u00e1n/n\u1ed1i v\u00e0o l\u00fac ch\u1ea1y). \"\n            \"H\u00e3y export l\u1ea1i b\u1eb1ng notebook, kh\u00f4ng ch\u1ec9nh s\u1eeda \u0111\u1ed3 th\u1ecb ONNX.\", \"unresolved_weights\", **detail)\n    diff_nodes = sig.conv_nodes - ref.conv_nodes\n    if diff_nodes > 0:\n        return errors.structure_mismatch(\n            f\"m\u00f4 h\u00ecnh c\u00f3 {sig.conv_nodes} l\u1edbp Conv, YOLOv8n c\u1ea7n \u0111\u00fang {ref.conv_nodes} (th\u1eeba {diff_nodes} l\u1edbp). \"\n            \"B\u1ea1n c\u00f3 th\u1ec3 \u0111\u00e3 th\u00eam l\u1edbp/\u0111\u1ea7u ra ph\u1ee5 ho\u1eb7c d\u00f9ng ki\u1ebfn tr\u00fac kh\u00e1c: h\u00e3y fine-tune t\u1eeb yolov8n.pt v\u00e0 export l\u1ea1i b\u1eb1ng notebook.\",\n            \"extra_layers\", **detail)\n    if diff_nodes < 0:\n        return errors.structure_mismatch(\n            f\"m\u00f4 h\u00ecnh ch\u1ec9 c\u00f3 {sig.conv_nodes} l\u1edbp Conv, YOLOv8n c\u1ea7n \u0111\u00fang {ref.conv_nodes} (thi\u1ebfu {-diff_nodes} l\u1edbp). \"\n            \"H\u00e3y fine-tune t\u1eeb yolov8n.pt v\u00e0 export l\u1ea1i b\u1eb1ng notebook.\", \"missing_layers\", **detail)\n    ref_stem, sub_stem = stem_channels(ref.triples), stem_channels(sig.triples)\n    if ref_stem and sub_stem and ref_stem != sub_stem:\n        ratio = sig.conv_params / max(ref.conv_params, 1)\n        return errors.structure_mismatch(\n            f\"l\u1edbp Conv \u0111\u1ea7u ti\u00ean c\u00f3 {sub_stem[0]} k\u00eanh (YOLOv8n: {ref_stem[0]}), t\u1ed5ng tham s\u1ed1 Conv g\u1ea5p {ratio:.1f} l\u1ea7n YOLOv8n \"\n            f\"({sig.conv_params / 1e6:.2f}M so v\u1edbi {ref.conv_params / 1e6:.2f}M). C\u00f3 v\u1ebb b\u1ea1n \u0111\u00e3 export YOLOv8s/m/l/x ho\u1eb7c m\u1ed9t ki\u1ebfn tr\u00fac \"\n            \"t\u00f9y ch\u1ec9nh: cu\u1ed9c thi ch\u1ec9 nh\u1eadn YOLOv8n, h\u00e3y fine-tune t\u1eeb yolov8n.pt v\u00e0 export l\u1ea1i.\", \"wrong_model_size\",\n            ref_stem=ref_stem, sub_stem=sub_stem, **detail)\n    shown = d[\"extra_tensors\"][:3] or d[\"missing_tensors\"][:3]\n    desc = \", \".join(f\"{'bias' if r else 'weight'} {_fmt_shape(s)} {dt}\" for r, dt, s in shown)\n    return errors.structure_mismatch(\n        f\"h\u00ecnh d\u1ea1ng tr\u1ecdng s\u1ed1 c\u00e1c l\u1edbp Conv kh\u00e1c v\u1edbi YOLOv8n chu\u1ea9n (v\u00ed d\u1ee5: {desc}). \"\n        \"H\u00e3y fine-tune t\u1eeb yolov8n.pt (kh\u00f4ng \u0111\u1ed5i ki\u1ebfn tr\u00fac) v\u00e0 export l\u1ea1i b\u1eb1ng notebook.\", \"weight_shapes\", **detail)\n",
  "validate.py": "\"\"\"ONNX submission validator.\n\n``validate_onnx(path, config)`` never raises for a bad submission: it returns a ``ValidationResult`` whose\n``code`` is one of ``errors.CODES``. Checks run in this order (cheapest and most informative first):\n\n 1. file size cap, SHA-256, parse WITHOUT loading external data (protobuf only)\n 2. resource guards (node / initializer / parameter counts, sparse initializers, local functions)\n 3. external-data references -> ONNX_EXTERNAL_DATA\n 4. custom op domains / functions -> ONNX_CUSTOM_OP\n 5. ``onnx.checker``\n 6. fingerprints (tensor + content) are computed so every later rejection still carries them\n 7. IO contract: one fp32 input [1,3,imgsz,imgsz], one fp32 output [1,4+nc,anchors]\n 8. fp32-only weights\n 9. reserved fingerprints (official pretrained export, provided baseline). This runs BEFORE the class-count\n    and structure checks because the pretrained model has nc=80 and would otherwise be reported as a\n    class-count error.\n10. class count (nc derived from the output channels vs the configuration)\n11. metadata ``names`` (if present) equal the configured classes, in order\n12. structure: op types inside the allowed set and the Conv weight signature equal to the reference\n\nThe ONNX graph topology itself is not verified beyond the Conv-weight multiset and the allowed op set\n(a known, documented gap).\n\"\"\"\nfrom __future__ import annotations\n\nimport ast\nimport json\nimport time\nfrom dataclasses import dataclass, field\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom typing import Any, Iterator\n\nimport numpy as np\nimport onnx\nfrom onnx import AttributeProto, TensorProto\n\nfrom . import errors\nfrom .config import Config, load_config\nfrom .fingerprint import dtype_name, is_non_fp32_float, model_tensor_fingerprint, sha256_file, tensor_bytes, tensor_shape\nfrom .signature import (\n    Reference, compute_signature, derive_nc, diagnose_mismatch, graph_inputs, iter_conv_params, tensor_type_dims,\n)\n\n_DATA_DIR = Path(__file__).resolve().parent / \"data\"\n_MAX_SUBGRAPH_DEPTH = 8\n_STD = frozenset((\"\", \"ai.onnx\"))  # standard operator domains\n\n\n@dataclass\nclass ValidationResult:\n    ok: bool\n    code: str | None\n    message_vi: str\n    fingerprint: str | None = None          # tensor fingerprint (design spec section 6)\n    content_fingerprint: str | None = None  # name-independent Conv-content fingerprint (stronger copy detector)\n    file_sha256: str | None = None          # sha256 of model.onnx, audit only\n    nc: int | None = None                   # class count derived from the model output\n    details: dict[str, Any] = field(default_factory=dict)\n\n    def to_dict(self) -> dict[str, Any]:\n        return {\n            \"ok\": self.ok, \"code\": self.code, \"message_vi\": self.message_vi,\n            \"fingerprint\": self.fingerprint, \"content_fingerprint\": self.content_fingerprint,\n            \"file_sha256\": self.file_sha256, \"nc\": self.nc, \"details\": self.details,\n        }\n\n\n# ------------------------------------------------------------------------------------------ shipped data\ndef _data_text(name: str) -> str:\n    # Resolved from __file__, not importlib.resources: when the process runs from the project root, the\n    # ``detscore/`` project directory can shadow the package as a namespace package.\n    return (_DATA_DIR / name).read_text(encoding=\"utf-8\")\n\n\n@lru_cache(maxsize=None)\ndef load_reference(path: str | None = None) -> Reference:\n    if path:\n        return Reference.load(path)\n    return Reference.from_json(json.loads(_data_text(\"reference_signature.json\")))\n\n\n@dataclass(frozen=True)\nclass ReservedEntry:\n    kind: str  # \"pretrained\" | \"baseline\"\n    label: str\n    tensor_fingerprint: str\n    content_fingerprint: str\n\n\n@lru_cache(maxsize=None)\ndef load_reserved(path: str | None = None) -> tuple[ReservedEntry, ...]:\n    text = Path(path).read_text(encoding=\"utf-8\") if path else _data_text(\"reserved_fingerprints.json\")\n    data = json.loads(text)\n    if data.get(\"schema_version\") != 1:\n        raise ValueError(\"unsupported reserved_fingerprints.json schema_version\")\n    return tuple(\n        ReservedEntry(e[\"kind\"], e.get(\"label\", e[\"kind\"]), e[\"tensor_fingerprint\"], e[\"content_fingerprint\"])\n        for e in data[\"entries\"]\n    )\n\n\n# ------------------------------------------------------------------------------------------ graph walking\ndef _iter_graphs(graph, depth: int = 0) -> Iterator[Any]:\n    if depth > _MAX_SUBGRAPH_DEPTH:\n        raise errors.onnx_parse(\"\u0111\u1ed3 th\u1ecb l\u1ed3ng nhau qu\u00e1 s\u00e2u\")\n    yield graph\n    for n in graph.node:\n        for a in n.attribute:\n            if a.type == AttributeProto.GRAPH:\n                yield from _iter_graphs(a.g, depth + 1)\n            elif a.type == AttributeProto.GRAPHS:\n                for g in a.graphs:\n                    yield from _iter_graphs(g, depth + 1)\n\n\ndef _iter_nodes(model) -> Iterator[Any]:\n    for g in _iter_graphs(model.graph):\n        yield from g.node\n\n\ndef _iter_tensors(model) -> Iterator[TensorProto]:\n    \"\"\"Every TensorProto in the model: initializers (all graphs) and tensor-valued node attributes.\"\"\"\n    for g in _iter_graphs(model.graph):\n        yield from g.initializer\n        for n in g.node:\n            for a in n.attribute:\n                if a.type == AttributeProto.TENSOR:\n                    yield a.t\n                elif a.type == AttributeProto.TENSORS:\n                    yield from a.tensors\n                elif a.type == AttributeProto.SPARSE_TENSOR:\n                    yield a.sparse_tensor.values\n                elif a.type == AttributeProto.SPARSE_TENSORS:\n                    for s in a.sparse_tensors:\n                        yield s.values\n\n\n# ------------------------------------------------------------------------------------------ the checks\ndef _parse(path: Path, cfg: Config) -> onnx.ModelProto:\n    size = path.stat().st_size\n    if size == 0:\n        raise errors.onnx_parse(\"t\u1ec7p r\u1ed7ng (0 byte)\")\n    if size > cfg.max_uncompressed_bytes:\n        raise errors.zip_too_large(size, cfg.max_uncompressed_bytes, what=\"T\u1ec7p model.onnx\")\n    data = path.read_bytes()\n    model = onnx.ModelProto()\n    try:\n        model.ParseFromString(data)  # protobuf only: external data is never loaded\n    except RecursionError as exc:\n        raise errors.onnx_parse(\"c\u1ea5u tr\u00fac protobuf l\u1ed3ng qu\u00e1 s\u00e2u\") from exc\n    except Exception as exc:  # noqa: BLE001 - google.protobuf.message.DecodeError and friends\n        raise errors.onnx_parse(f\"kh\u00f4ng ph\u1ea3i \u0111\u1ecbnh d\u1ea1ng ONNX h\u1ee3p l\u1ec7 ({str(exc)[:100] or type(exc).__name__})\") from exc\n    if not model.HasField(\"graph\") or not model.graph.node:\n        raise errors.onnx_parse(\"t\u1ec7p kh\u00f4ng ch\u1ee9a \u0111\u1ed3 th\u1ecb t\u00ednh to\u00e1n n\u00e0o (c\u00f3 th\u1ec3 b\u1ecb c\u1eaft c\u1ee5t ho\u1eb7c r\u1ed7ng)\")\n    return model\n\n\ndef _guards(model: onnx.ModelProto, cfg: Config) -> int:\n    \"\"\"Resource guards; returns the total number of initializer elements.\"\"\"\n    n_nodes = sum(1 for _ in _iter_nodes(model))\n    if n_nodes > cfg.max_nodes:\n        raise errors.structure_mismatch(\n            f\"m\u00f4 h\u00ecnh c\u00f3 {n_nodes} node, v\u01b0\u1ee3t gi\u1edbi h\u1ea1n {cfg.max_nodes} (YOLOv8n export ch\u1ec9 c\u00f3 kho\u1ea3ng 230-450 node).\",\n            \"too_many_nodes\", n_nodes=n_nodes, limit=cfg.max_nodes)\n    n_init = len(model.graph.initializer)\n    if n_init > cfg.max_initializers:\n        raise errors.structure_mismatch(\n            f\"m\u00f4 h\u00ecnh c\u00f3 {n_init} tensor h\u1eb1ng s\u1ed1, v\u01b0\u1ee3t gi\u1edbi h\u1ea1n {cfg.max_initializers}.\",\n            \"too_many_initializers\", n_initializers=n_init, limit=cfg.max_initializers)\n    if len(model.graph.sparse_initializer):\n        raise errors.onnx_parse(\"m\u00f4 h\u00ecnh d\u00f9ng sparse initializer, kh\u00f4ng \u0111\u01b0\u1ee3c h\u1ed7 tr\u1ee3\")\n    total = 0\n    for t in _iter_tensors(model):\n        if t.data_location == TensorProto.EXTERNAL or len(t.external_data):\n            raise errors.onnx_external_data(t.name or \"(kh\u00f4ng t\u00ean)\")\n        n = 1\n        for d in tensor_shape(t, cfg.max_total_params):\n            n *= d\n        total += n\n        if total > cfg.max_total_params:\n            raise errors.structure_mismatch(\n                f\"t\u1ed5ng s\u1ed1 tham s\u1ed1 v\u01b0\u1ee3t gi\u1edbi h\u1ea1n {cfg.max_total_params / 1e6:.0f}M ph\u1ea7n t\u1eed (YOLOv8n kho\u1ea3ng 3.2M). \"\n                \"Cu\u1ed9c thi ch\u1ec9 nh\u1eadn YOLOv8n.\", \"too_many_params\", total=total, limit=cfg.max_total_params)\n    return total\n\n\ndef _check_custom_ops(model: onnx.ModelProto) -> None:\n    found: list[str] = []\n    for o in model.opset_import:\n        if o.domain not in _STD:\n            found.append(f\"opset domain '{o.domain}'\")\n    for n in _iter_nodes(model):\n        if n.domain not in _STD:\n            found.append(f\"{n.op_type} (domain '{n.domain}')\")\n    if len(model.functions):\n        found.extend(f\"h\u00e0m c\u1ee5c b\u1ed9 '{f.name}'\" for f in model.functions)\n    if found:\n        raise errors.onnx_custom_op(list(dict.fromkeys(found)))\n\n\n\ndef _run_checker(model: onnx.ModelProto) -> None:\n    try:\n        onnx.checker.check_model(model)\n    except Exception as exc:  # noqa: BLE001 - onnx.checker.ValidationError and friends\n        text = str(exc).strip().splitlines()\n        reason = \"onnx.checker t\u1eeb ch\u1ed1i: \" + ((text[0][:160].rstrip(\" :\") + \".\") if text else type(exc).__name__)\n        half = any(t.data_type == TensorProto.FLOAT16 for t in model.graph.initializer)\n        hint = (\"M\u00f4 h\u00ecnh c\u00f3 tr\u1ecdng s\u1ed1 fp16: \u0111\u00e2y th\u01b0\u1eddng l\u00e0 k\u1ebft qu\u1ea3 c\u1ee7a export v\u1edbi half=True. \"\n                \"H\u00e3y export v\u1edbi half=False (\u0111\u00fang nh\u01b0 \u00f4 export trong notebook).\") if half else \"\"\n        raise errors.onnx_parse(reason, hint) from exc\n\n\ndef _fmt_dims(dims: list[Any]) -> str:\n    return \"[\" + \", \".join(\"?\" if d is None else str(d) for d in dims) + \"]\"\n\n\ndef _check_io(model: onnx.ModelProto, cfg: Config) -> tuple[list[Any], list[Any]]:\n    inputs = graph_inputs(model)\n    if len(inputs) != 1:\n        raise errors.onnx_io(\"S\u1ed1 input\", f\"{len(inputs)} input ({', '.join(v.name for v in inputs) or 'kh\u00f4ng c\u00f3'})\", \"\u0111\u00fang 1 input (images)\")\n    et, in_dims = tensor_type_dims(inputs[0])\n    if et != TensorProto.FLOAT:\n        raise errors.onnx_dtype(\"Input\", dtype_name(et))\n    if tuple(in_dims) != cfg.expected_input_shape:\n        dyn = any(not isinstance(d, int) for d in in_dims)\n        hint = \"K\u00edch th\u01b0\u1edbc input ph\u1ea3i c\u1ed1 \u0111\u1ecbnh: h\u00e3y export v\u1edbi dynamic=False v\u00e0 imgsz=640.\" if dyn else \"H\u00e3y export v\u1edbi imgsz=640, batch=1.\"\n        raise errors.onnx_io(\"Input\", _fmt_dims(in_dims), _fmt_dims(list(cfg.expected_input_shape)), hint)\n\n    outs = list(model.graph.output)\n    if len(outs) != 1:\n        raise errors.onnx_io(\"S\u1ed1 output\", f\"{len(outs)} output ({', '.join(v.name for v in outs) or 'kh\u00f4ng c\u00f3'})\",\n                             \"\u0111\u00fang 1 output [1, 4+nc, 8400]\", \"\u0110\u1eebng b\u1eadt nms=True hay end2end khi export.\")\n    et, out_dims = tensor_type_dims(outs[0])\n    if et != TensorProto.FLOAT:\n        raise errors.onnx_dtype(\"Output\", dtype_name(et))\n    anchors = cfg.num_anchors\n    expected_txt = f\"[1, 4+nc, {anchors}]\"\n    if len(out_dims) != 3 or out_dims[0] != 1 or out_dims[2] != anchors or derive_nc(out_dims) is None:\n        hint = (\"\u0110\u1eebng b\u1eadt nms=True/end2end khi export: cu\u1ed9c thi c\u1ea7n output th\u00f4 [1, 4+nc, 8400]; h\u1ec7 th\u1ed1ng t\u1ef1 ch\u1ea1y NMS.\"\n                if len(out_dims) == 3 and out_dims[2] != anchors else\n                \"H\u00e3y export v\u1edbi dynamic=False, imgsz=640, nms=False.\")\n        raise errors.onnx_io(\"Output\", _fmt_dims(out_dims), expected_txt, hint)\n    return in_dims, out_dims\n\n\ndef _check_fp32(model: onnx.ModelProto) -> None:\n    bad: dict[str, int] = {}\n    for t in _iter_tensors(model):\n        if is_non_fp32_float(t.data_type):\n            bad[dtype_name(t.data_type)] = bad.get(dtype_name(t.data_type), 0) + 1\n    for n in _iter_nodes(model):\n        if n.op_type == \"Cast\":\n            for a in n.attribute:\n                if a.name == \"to\" and is_non_fp32_float(int(a.i)):\n                    bad[dtype_name(int(a.i))] = bad.get(dtype_name(int(a.i)), 0) + 1\n    if bad:\n        found = \", \".join(f\"{k} ({v} tensor/cast)\" for k, v in sorted(bad.items()))\n        raise errors.onnx_dtype(\"Tr\u1ecdng s\u1ed1 m\u00f4 h\u00ecnh\", found)\n\n\n_FLOAT_NUMPY = {\"float16\": \"<f2\", \"float32\": \"<f4\", \"float64\": \"<f8\"}\n\n\ndef _check_weights(model: onnx.ModelProto, cfg: Config) -> None:\n    \"\"\"Reject a checkpoint whose Conv weights/biases hold NaN/Inf, or whose Conv kernels are all exactly zero.\n\n    Such a model is not a trained model, but every team whose training diverged (or who exported an untouched\n    zero-initialised network) would produce a bit-identical one, so the duplicate-checkpoint fingerprint would\n    flag unrelated teams as copiers of each other. Rejecting it here (no quota charge) removes the false positive at\n    its source. Unresolved Conv parameters are left to the structure check.\n    \"\"\"\n    bad = total = bad_tensors = 0\n    kernels = nonzero_kernels = 0\n    for role, t in iter_conv_params(model):\n        if t is None:\n            continue\n        dt, _shape, raw = tensor_bytes(t, cfg.max_total_params)\n        np_dtype = _FLOAT_NUMPY.get(dt)\n        if np_dtype is None:\n            continue  # non-float (or bfloat16): the dtype checks own this case\n        arr = np.frombuffer(raw, dtype=np_dtype)\n        total += arr.size\n        n_bad = int(arr.size - np.count_nonzero(np.isfinite(arr)))\n        if n_bad:\n            bad += n_bad\n            bad_tensors += 1\n        if role == 0:\n            kernels += 1\n            nonzero_kernels += int(np.any(arr != 0))\n    if bad:\n        raise errors.invalid_weights(\"nonfinite\", bad_elements=bad, total_elements=total, bad_tensors=bad_tensors)\n    if kernels and not nonzero_kernels:\n        raise errors.invalid_weights(\"all_zero\", conv_kernels=kernels)\n\n\ndef _check_reserved(entries: tuple[ReservedEntry, ...], tensor_fp: str, content_fp: str) -> None:\n    for e in entries:\n        if tensor_fp == e.tensor_fingerprint or content_fp == e.content_fingerprint:\n            raise errors.pretrained_model() if e.kind == \"pretrained\" else errors.baseline_model()\n\n\ndef _parse_names(value: str, cfg: Config) -> list[str] | None:\n    if len(value) > cfg.max_metadata_chars:\n        return None\n    obj: Any\n    try:\n        obj = ast.literal_eval(value)\n    except (ValueError, SyntaxError, MemoryError, RecursionError):\n        try:\n            obj = json.loads(value)\n        except (ValueError, RecursionError):\n            return None\n    if isinstance(obj, dict):\n        try:\n            keys = sorted(int(k) for k in obj)\n        except (TypeError, ValueError):\n            return None\n        by_key = {int(k): v for k, v in obj.items()}\n        if keys != list(range(len(keys))) or not all(isinstance(by_key[k], str) for k in keys):\n            return None\n        return [by_key[k] for k in keys]\n    if isinstance(obj, (list, tuple)) and all(isinstance(x, str) for x in obj):\n        return list(obj)\n    return None\n\n\ndef _check_names(model: onnx.ModelProto, cfg: Config) -> bool:\n    \"\"\"Returns True when a ``names`` metadata entry exists (and matches).\"\"\"\n    value = None\n    for p in model.metadata_props:\n        if p.key == \"names\":\n            value = p.value\n    if value is None:\n        return False\n    expected = list(cfg.classes)\n    names = _parse_names(value, cfg)\n    if names is None:\n        raise errors.meta_names(expected, value[:200], \"kh\u00f4ng \u0111\u1ecdc \u0111\u01b0\u1ee3c \u0111\u1ecbnh d\u1ea1ng\")\n    if names != expected:\n        if sorted(names) == sorted(expected):\n            why = \"\u0111\u00fang t\u00ean nh\u01b0ng sai th\u1ee9 t\u1ef1 (id l\u1edbp ph\u1ea3i kh\u1edbp th\u1ee9 t\u1ef1 trong cu\u1ed9c thi)\"\n        elif len(names) != len(expected):\n            why = f\"c\u00f3 {len(names)} t\u00ean l\u1edbp, c\u1ea7n {len(expected)}\"\n        else:\n            why = \"t\u00ean l\u1edbp kh\u00e1c v\u1edbi danh s\u00e1ch c\u1ee7a cu\u1ed9c thi\"\n        raise errors.meta_names(expected, names, why)\n    return True\n\n\ndef _check_structure(model: onnx.ModelProto, sig, ref: Reference, cfg: Config) -> None:\n    if ref.nc != cfg.nc or ref.imgsz != cfg.imgsz:\n        raise errors.runtime_error(\n            f\"reference_signature.json \u0111\u01b0\u1ee3c t\u1ea1o cho nc={ref.nc}, imgsz={ref.imgsz} nh\u01b0ng c\u1ea5u h\u00ecnh cu\u1ed9c thi l\u00e0 \"\n            f\"nc={cfg.nc}, imgsz={cfg.imgsz}; admin c\u1ea7n ch\u1ea1y tools/make_reference.py\")\n    extra_ops = sorted(set(sig.op_types) - set(ref.allowed_op_types))\n    if extra_ops:\n        raise errors.structure_mismatch(\n            f\"m\u00f4 h\u00ecnh ch\u1ee9a lo\u1ea1i l\u1edbp/to\u00e1n t\u1eed kh\u00f4ng c\u00f3 trong YOLOv8n: {', '.join(extra_ops)}. \"\n            \"\u0110\u00e2y l\u00e0 c\u00e1c l\u1edbp th\u1eeba so v\u1edbi export chu\u1ea9n; h\u00e3y fine-tune t\u1eeb yolov8n.pt v\u00e0 export l\u1ea1i b\u1eb1ng notebook, kh\u00f4ng s\u1eeda \u0111\u1ed3 th\u1ecb.\",\n            \"extra_op_types\", extra_op_types=extra_ops)\n    if sig.digest != ref.signature_sha256 or sig.conv_nodes != ref.conv_nodes:\n        raise diagnose_mismatch(ref, sig)\n\n\n# ------------------------------------------------------------------------------------------ entry point\ndef validate_onnx(model_path: str | Path, config: Config | None = None, *, reference: Reference | None = None,\n                  reserved: tuple[ReservedEntry, ...] | None = None) -> ValidationResult:\n    \"\"\"Validate one ``model.onnx``. Returns a result; never raises ``SubmissionError``.\"\"\"\n    cfg = config or load_config()\n    path = Path(model_path)\n    timings: dict[str, float] = {}\n    state: dict[str, Any] = {\"file_sha256\": None, \"fingerprint\": None, \"content_fingerprint\": None, \"nc\": None}\n    details: dict[str, Any] = {\"timings_ms\": timings}\n\n    def tick(name: str, t0: float) -> None:\n        timings[name] = round((time.perf_counter() - t0) * 1000, 1)\n\n    try:\n        t = time.perf_counter()\n        try:\n            size = path.stat().st_size\n        except OSError as exc:\n            raise errors.onnx_parse(f\"kh\u00f4ng m\u1edf \u0111\u01b0\u1ee3c t\u1ec7p: {exc.strerror or exc}\") from exc\n        details[\"bytes\"] = size\n        if size <= cfg.max_uncompressed_bytes:\n            state[\"file_sha256\"] = sha256_file(path)\n        model = _parse(path, cfg)\n        tick(\"parse\", t)\n\n        t = time.perf_counter()\n        total_params = _guards(model, cfg)\n        _check_custom_ops(model)\n        tick(\"guards\", t)\n\n        t = time.perf_counter()\n        _run_checker(model)\n        tick(\"checker\", t)\n\n        t = time.perf_counter()\n        state[\"fingerprint\"] = model_tensor_fingerprint(model, cfg.max_total_params)\n        sig = compute_signature(model, cfg.max_total_params)\n        state[\"content_fingerprint\"] = sig.content_fingerprint\n        tick(\"fingerprint\", t)\n\n        in_dims, out_dims = _check_io(model, cfg)\n        found_nc = derive_nc(out_dims)\n        state[\"nc\"] = found_nc\n        _check_fp32(model)\n        _check_weights(model, cfg)\n\n        _check_reserved(reserved if reserved is not None else load_reserved(), state[\"fingerprint\"], state[\"content_fingerprint\"])\n\n        if found_nc != cfg.nc:\n            raise errors.structure_class_count(found_nc, cfg.nc, cfg.classes)\n        names_present = _check_names(model, cfg)\n\n        _check_structure(model, sig, reference or load_reference(), cfg)\n        details.update({\n            \"input_shape\": in_dims, \"output_shape\": out_dims, \"names_metadata_present\": names_present,\n            \"n_nodes\": sum(op for op in sig.op_types.values()), \"conv_nodes\": sig.conv_nodes,\n            \"conv_params\": sig.conv_params, \"total_params\": total_params, \"signature_sha256\": sig.digest,\n            \"n_initializers\": len(model.graph.initializer), \"ir_version\": int(model.ir_version),\n            \"opset\": {(o.domain or \"ai.onnx\"): int(o.version) for o in model.opset_import},\n        })\n        return ValidationResult(True, None, \"T\u1ec7p ONNX h\u1ee3p l\u1ec7.\", state[\"fingerprint\"], state[\"content_fingerprint\"],\n                                state[\"file_sha256\"], found_nc, details)\n    except errors.SubmissionError as exc:\n        details[\"error_detail\"] = exc.detail\n        return ValidationResult(False, exc.code, exc.message_vi, state[\"fingerprint\"], state[\"content_fingerprint\"],\n                                state[\"file_sha256\"], state[\"nc\"], details)\n    except MemoryError:\n        exc = errors.memory_limit(cfg.container_memory_mb)\n        return ValidationResult(False, exc.code, exc.message_vi, state[\"fingerprint\"], state[\"content_fingerprint\"],\n                                state[\"file_sha256\"], state[\"nc\"], details)\n    except RecursionError:\n        exc = errors.onnx_parse(\"c\u1ea5u tr\u00fac t\u1ec7p l\u1ed3ng qu\u00e1 s\u00e2u\")\n        return ValidationResult(False, exc.code, exc.message_vi, state[\"fingerprint\"], state[\"content_fingerprint\"],\n                                state[\"file_sha256\"], state[\"nc\"], details)\n    except Exception as exc:  # noqa: BLE001 - last resort: report, never crash the container\n        err = errors.runtime_error(f\"{type(exc).__name__}: {str(exc)[:160]}\")\n        details[\"error_detail\"] = err.detail\n        return ValidationResult(False, err.code, err.message_vi, state[\"fingerprint\"], state[\"content_fingerprint\"],\n                                state[\"file_sha256\"], state[\"nc\"], details)\n"
 }
}''')

print(f"Notebook {NOTEBOOK_VERSION} | lớp: {', '.join(CLASSES)} | ultralytics=={ULTRALYTICS_VERSION}")

In [ ]:
#@title Thư viện: mã số đội, xuất ONNX, kiểm tra và đóng gói bài nộp (không cần chỉnh sửa) { display-mode: "form" }
# Cell source of the simplified notebook. Definitions only (no side effects), so the unit tests exec the same text.
# The self-check is the server's own validator, inlined at build time (SELFCHECK_BUNDLE), so the notebook can never
# accept a file the grading system would reject.
import hashlib
import importlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = ("google.colab" in sys.modules) or ("COLAB_RELEASE_TAG" in os.environ) or ("COLAB_GPU" in os.environ)
ZIP_ENTRY_NAME = "model.onnx"
EXPORT_KEYS = ("format", "imgsz", "opset", "simplify", "dynamic", "half", "nms", "batch")


class NotebookError(Exception):
    """Every student-facing error. ``message_vi`` is what the student reads."""

    def __init__(self, message_vi, code="NOTEBOOK"):
        super().__init__(message_vi)
        self.message_vi = message_vi
        self.code = code


class SelfCheckError(NotebookError):
    """The exported model would be rejected by the server."""

    def __init__(self, message_vi, result=None):
        super().__init__(message_vi, "SELFCHECK")
        self.result = result


# ------------------------------------------------------------------------------------------------ team id / seed
def validate_team_id(team_id):
    """TEAM_ID must be a non-negative integer: the seed of the team, counted up from 0 by the website (see the account
    e-mail and the Data tab). The cell's default -1 means "not filled in yet"."""
    fill = "Hãy điền đúng seed của đội do ban tổ chức cấp (số nguyên từ 0 trở lên, có trong email tài khoản và ở tab Dữ liệu của website) vào ô «Mã số đội» rồi chạy lại."
    if isinstance(team_id, bool) or not isinstance(team_id, int):
        raise NotebookError(f"Mã số đội (TEAM_ID) phải là một số nguyên không âm, ví dụ 7 (đang là {team_id!r}). {fill}", "TEAM_ID")
    if team_id < 0:
        raise NotebookError(f"Mã số đội (TEAM_ID = {team_id}) chưa hợp lệ. {fill}", "TEAM_ID")
    if team_id > 2 ** 31 - 1:
        raise NotebookError(f"Mã số đội (TEAM_ID = {team_id}) quá lớn. {fill}", "TEAM_ID")
    return team_id


def seed_everything(seed):
    """Seed python ``random``, numpy and (when importable) torch. Returns what was seeded."""
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    seeded = {"python_random": True, "numpy": False, "torch": False}
    for name in ("numpy", "torch"):
        try:
            mod = importlib.import_module(name)
        except Exception:  # noqa: BLE001 - not installed yet: nothing to seed
            continue
        if name == "numpy":
            mod.random.seed(seed)
        else:
            mod.manual_seed(seed)
            if mod.cuda.is_available():
                mod.cuda.manual_seed_all(seed)
        seeded[name] = True
    return seeded


def install_requirements(specs, runner=subprocess.run):
    """pip install the pinned packages (plain subprocess, works as a normal cell and as a script)."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *specs]
    proc = runner(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        tail = "\n".join(((proc.stdout or "") + (proc.stderr or "")).strip().splitlines()[-12:])
        raise NotebookError(
            f"Cài đặt thư viện thất bại (mã lỗi {proc.returncode}).\n{tail}\n"
            "Kiểm tra mạng, rồi Runtime > Disconnect and delete runtime và chạy lại từ ô đầu tiên.", "SETUP")
    return cmd


# ------------------------------------------------------------------------------------------------ export contract
def export_kwargs(contract=None):
    """Exact keyword arguments of ``YOLO(best).export`` taken from the server's export contract."""
    contract = EXPORT_CONTRACT if contract is None else contract
    return {k: contract[k] for k in EXPORT_KEYS}


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ------------------------------------------------------------------------------------------------ self-check
def load_selfcheck(bundle, dest_root):
    """Write the inlined detscore sources as the package ``detscore_selfcheck`` and import it.

    The package name differs from the server's ``detscore`` on purpose so it can never shadow an installed copy.
    Returns (validate_module, package_module, config).
    """
    pkg = Path(dest_root) / "detscore_selfcheck"
    if pkg.exists():
        shutil.rmtree(pkg)
    for rel, text in bundle["files"].items():
        target = pkg / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(text, encoding="utf-8")
    (pkg / "competition_config.yaml").write_text(bundle["config_yaml"], encoding="utf-8")
    for name in [m for m in sys.modules if m == "detscore_selfcheck" or m.startswith("detscore_selfcheck.")]:
        del sys.modules[name]
    root = str(Path(dest_root))
    if root not in sys.path:
        sys.path.insert(0, root)
    importlib.invalidate_caches()
    validate = importlib.import_module("detscore_selfcheck.validate")
    package = importlib.import_module("detscore_selfcheck.package")
    config = importlib.import_module("detscore_selfcheck.config").load_config(pkg / "competition_config.yaml")
    return validate, package, config


_SANITY_SCRIPT = """
import json, sys
import numpy as np
try:
    import onnxruntime as ort
except ImportError:
    print(json.dumps({"status": "skipped"}))
    raise SystemExit(0)
model, shape = sys.argv[1], json.loads(sys.argv[2])
opts = ort.SessionOptions()
opts.intra_op_num_threads = 1
sess = ort.InferenceSession(model, opts, providers=["CPUExecutionProvider"])
(y,) = sess.run(None, {sess.get_inputs()[0].name: np.full(shape, 0.5, dtype=np.float32)})
print(json.dumps({"status": "ran", "shape": list(y.shape), "finite": bool(np.isfinite(y).all())}))
"""


def sanity_forward(model_path, config, timeout_s=300):
    """One forward pass on a gray image in a child process: output must be finite and shaped [1, 4+nc, 8400]."""
    cmd = [sys.executable, "-c", _SANITY_SCRIPT, str(model_path), json.dumps(list(config.expected_input_shape))]
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s)
    except subprocess.TimeoutExpired as exc:
        raise SelfCheckError(f"TIME_LIMIT: chạy thử mô hình bằng onnxruntime quá {timeout_s} giây.") from exc
    lines = [ln for ln in proc.stdout.strip().splitlines() if ln.strip()]
    if proc.returncode != 0 or not lines:
        tail = (proc.stderr.strip().splitlines() or ["không rõ lỗi"])[-1][:160]
        raise SelfCheckError(f"RUNTIME_ERROR: onnxruntime không chạy được mô hình ({tail}).")
    out = json.loads(lines[-1])
    if out["status"] == "skipped":
        return {"status": "skipped", "reason": "onnxruntime không có"}
    expected = list(config.expected_output_shape)
    if out["shape"] != expected or not out["finite"]:
        raise SelfCheckError(f"RUNTIME_ERROR: output thực tế {out['shape']} hoặc chứa NaN/Inf; cần {expected} hữu hạn.")
    return {"status": "ok", "output_shape": out["shape"]}


def run_selfcheck(model_path, bundle, work_dir, printer=print):
    """Validate ``model_path`` exactly as the server's sandbox would. Raises SelfCheckError when it fails."""
    validate, _package, config = load_selfcheck(bundle, Path(work_dir) / "_selfcheck")
    res = validate.validate_onnx(model_path, config)
    if not res.ok:
        printer("\nKIỂM TRA THẤT BẠI: hệ thống chấm sẽ từ chối tệp này với thông báo sau.")
        printer(f"  Mã lỗi: {res.code}")
        printer(f"  {res.message_vi}")
        raise SelfCheckError(f"{res.code}: {res.message_vi}", res)
    info = res.to_dict()
    info["sanity_forward"] = sanity_forward(model_path, config)
    printer(f"Kiểm tra ĐẠT: cấu trúc YOLOv8n, input/output, fp32, tên lớp và dấu vân tay hợp lệ (nc={res.nc}).")
    return info


# ------------------------------------------------------------------------------------------------ zip
def write_submission_zip(model_path, zip_path):
    """submission.zip with exactly one entry ``model.onnx`` (fixed timestamp so the zip is reproducible)."""
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()
    info = zipfile.ZipInfo(ZIP_ENTRY_NAME, date_time=(1980, 1, 1, 0, 0, 0))
    info.compress_type = zipfile.ZIP_DEFLATED
    info.external_attr = 0o644 << 16
    with zipfile.ZipFile(zip_path, "w") as zf:
        zf.writestr(info, Path(model_path).read_bytes())
    return zip_path


def verify_submission_zip(zip_path, bundle, work_dir, expected_sha256):
    """Re-open the zip with the server's package checks and compare the model bytes. Returns a dict."""
    _validate, package, config = load_selfcheck(bundle, Path(work_dir) / "_selfcheck")
    sub_error = importlib.import_module("detscore_selfcheck.errors").SubmissionError
    extract_dir = Path(work_dir) / "_zipcheck"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    try:
        summary = package.inspect_zip(str(zip_path), config)
        out = package.extract_model(str(zip_path), extract_dir, config)
    except sub_error as exc:
        raise SelfCheckError(f"{exc.code}: {exc.message_vi}") from exc
    if sha256_file(out) != expected_sha256:
        raise SelfCheckError("ZIP_CORRUPT: model.onnx trong zip khác tệp đã kiểm tra. Hãy chạy lại ô này.")
    with zipfile.ZipFile(zip_path) as zf:
        entries = zf.namelist()
    return {"entries": entries, "zip_summary": summary.to_dict()}


# ------------------------------------------------------------------------------------------------ one call
def make_submission(weights, work_dir, YOLO, team_id, seed, printer=print):
    """Export ``weights`` (a YOLOv8n .pt with one class) -> model.onnx -> validate -> submission.zip -> verify -> report.

    Returns {"zip": Path, "report": Path, "info": dict}. Raises NotebookError / SelfCheckError with a Vietnamese message
    and leaves NO submission.zip behind when anything fails.
    """
    weights = Path(weights)
    if weights.suffix != ".pt" or not weights.is_file() or weights.stat().st_size == 0:
        raise NotebookError(f"Không tìm thấy tệp trọng số hợp lệ: {weights}. Hãy chạy ô huấn luyện trước (hoặc đặt "
                            "BEST_WEIGHTS tới tệp .pt của YOLOv8n 1 lớp).", "WEIGHTS")
    out_dir = Path(work_dir) / "submission"
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    kwargs = export_kwargs()
    printer(f"Xuất ONNX với tham số cố định: {kwargs}")
    exported = YOLO(str(weights)).export(**kwargs)
    exported = Path(str(exported))
    if not exported.is_file():
        raise NotebookError("Ultralytics không tạo được tệp ONNX. Xem lỗi phía trên và chạy lại ô này.", "EXPORT")
    model_path = out_dir / ZIP_ENTRY_NAME
    shutil.copyfile(exported, model_path)

    info = run_selfcheck(model_path, SELFCHECK_BUNDLE, work_dir, printer)
    model_sha = sha256_file(model_path)
    zip_path = write_submission_zip(model_path, out_dir / "submission.zip")
    try:
        zip_info = verify_submission_zip(zip_path, SELFCHECK_BUNDLE, work_dir, model_sha)
    except NotebookError:
        zip_path.unlink(missing_ok=True)
        raise

    report = {
        "notebook": {"name": NOTEBOOK_NAME, "version": NOTEBOOK_VERSION, "detscore_version": DETSCORE_VERSION,
                     "pinned_ultralytics": ULTRALYTICS_VERSION},
        "team_id": team_id,
        "seed": seed,
        "classes": list(CLASSES),
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "environment": {"in_colab": IN_COLAB, "python": platform.python_version(), "platform": platform.platform()},
        "export_args": kwargs,
        "model": {"sha256": model_sha, "bytes": model_path.stat().st_size, "fingerprint": info.get("fingerprint"),
                  "nc": info.get("nc")},
        "selfcheck": {"ok": True, "sanity_forward": info.get("sanity_forward")},
        "submission": {"zip_bytes": zip_path.stat().st_size, "entries": zip_info["entries"]},
    }
    report_path = out_dir / "run_report.json"
    report_path.write_text(json.dumps(report, indent=1, ensure_ascii=False), encoding="utf-8")

    printer("\n" + "=" * 64)
    printer(" BÀI NỘP HỢP LỆ: hệ thống chấm sẽ chấp nhận tệp này.")
    printer(f"   Tệp nộp : {zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB, đúng 1 mục: {ZIP_ENTRY_NAME})")
    printer(f"   Mã số đội (seed): {team_id}")
    printer("=" * 64)
    return {"zip": zip_path, "report": report_path, "info": info}


def offer_download(paths):
    """Colab: download the files to the student's machine. Elsewhere: just print where they are."""
    for p in paths:
        print(f"Tệp: {p}")
    if IN_COLAB:
        from google.colab import files  # noqa: PLC0415 - only exists in Colab

        for p in paths:
            files.download(str(p))

## Bước 0. Cài đặt và mã số đội
Ô đầu cài đúng phiên bản `ultralytics` của cuộc thi. Ô thứ hai: điền **seed của đội** (số nguyên từ 0 trở lên, có trong email tài khoản và ở tab Dữ liệu của website; ví dụ đội đầu tiên có seed 0). Số này là **seed**
cho mọi thứ ngẫu nhiên (chia train/val, huấn luyện), nên cùng dữ liệu và cùng mã số đội sẽ cho kết quả gần như giống nhau khi chạy lại.

In [ ]:
install_requirements(INSTALL_SPECS)
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ. Hãy bật GPU: Runtime > Change runtime type > T4 GPU")

In [ ]:
#@title Mã số đội (bắt buộc) { display-mode: "form" }
TEAM_ID = 23  #@param {type:"integer"}

SEED = validate_team_id(TEAM_ID)  # lỗi tiếng Việt nếu chưa điền (-1 = chưa điền)
seed_everything(SEED)
print(f"Mã số đội = {TEAM_ID}, seed = {SEED}")

## Bước 1. Dữ liệu của đội (bạn được sửa tự do)
Chuẩn bị **một tệp .zip** dữ liệu YOLO Detection: ảnh và nhãn `.txt` (mỗi dòng `0 cx cy w h`, tọa độ chuẩn hóa trong [0, 1]).
Hàm mẫu bên dưới tự tìm ảnh và nhãn, kiểm tra nhãn cơ bản (chỉ có lớp id 0 = `GreenSM`), chia train/val (80/20, theo seed) nếu dữ liệu chưa chia sẵn,
rồi tạo `data.yaml`. Ảnh không có tệp nhãn được coi là ảnh nền (không có xe GreenSM). Bạn có thể thay bằng cách nạp dữ liệu của riêng đội.

Đưa zip vào bằng đường dẫn trên Google Drive (nhanh hơn), hoặc để trống để hiện nút tải lên.

In [ ]:
#@markdown Đường dẫn tới zip dữ liệu trên Drive, ví dụ `/content/drive/MyDrive/du_lieu.zip` (để trống để tải lên trực tiếp):
DATASET_ZIP_PATH = ""  #@param {type:"string"}

WORK_DIR = Path("/content/work") if IN_COLAB else Path("work")
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def get_dataset_zip(path_text):
    if path_text.strip():
        if IN_COLAB and path_text.startswith("/content/drive"):
            from google.colab import drive  # noqa: PLC0415

            drive.mount("/content/drive")
        zip_path = Path(path_text.strip())
    elif IN_COLAB:
        from google.colab import files  # noqa: PLC0415

        uploaded = files.upload()
        zip_path = Path(next(iter(uploaded)))
    else:
        raise NotebookError("Hãy điền DATASET_ZIP_PATH (đường dẫn tới zip dữ liệu).", "DATA")
    if not zip_path.is_file():
        raise NotebookError(f"Không tìm thấy tệp {zip_path}. Kiểm tra lại đường dẫn.", "DATA")
    return zip_path


def label_path_of(img):
    """images/a/x.jpg -> labels/a/x.txt; otherwise x.txt next to the image; None when there is no label file."""
    parts = list(img.parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images")
        parts[i] = "labels"
        cand = Path(*parts).with_suffix(".txt")
        if cand.is_file():
            return cand
    cand = img.with_suffix(".txt")
    return cand if cand.is_file() else None


def read_labels(label_file):
    """Return (lines, problems). A valid line is `0 cx cy w h` with coordinates in [0, 1]."""
    lines, problems = [], []
    if label_file is None:
        return lines, problems
    for n, line in enumerate(label_file.read_text(encoding="utf-8", errors="replace").splitlines(), 1):
        if not line.strip():
            continue
        cols = line.split()
        if len(cols) != 5:
            problems.append(f"{label_file.name} dòng {n}: cần đúng 5 cột `0 cx cy w h` (đang có {len(cols)})")
            continue
        try:
            cls, vals = int(cols[0]), [float(x) for x in cols[1:]]
        except ValueError:
            problems.append(f"{label_file.name} dòng {n}: giá trị không phải số")
            continue
        if cls != 0:
            problems.append(f"{label_file.name} dòng {n}: id lớp {cls} không hợp lệ, cuộc thi chỉ có 1 lớp {CLASSES[0]} (id 0)")
        elif not all(0.0 <= v <= 1.0 for v in vals):
            problems.append(f"{label_file.name} dòng {n}: tọa độ phải nằm trong [0, 1]")
        else:
            lines.append(" ".join(cols))
    return lines, problems


def prepare_dataset(zip_path, out_dir, seed, val_ratio=0.2):
    out_dir = Path(out_dir)
    raw_dir = out_dir.parent / "raw"
    for d in (out_dir, raw_dir):
        shutil.rmtree(d, ignore_errors=True)
    raw_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(raw_dir)
    images = sorted(p for p in raw_dir.rglob("*") if p.suffix.lower() in IMG_EXTS and "__MACOSX" not in p.parts)
    if len(images) < 2:
        raise NotebookError(f"Cần ít nhất 2 ảnh trong zip (tìm thấy {len(images)}).", "DATA")

    items, problems, n_boxes = [], [], 0
    for img in images:
        lines, probs = read_labels(label_path_of(img))
        problems += probs
        n_boxes += len(lines)
        items.append((img, lines))
    if problems:
        extra = f"\n  ... và {len(problems) - 10} lỗi khác" if len(problems) > 10 else ""
        raise NotebookError("Nhãn có lỗi, hãy sửa rồi tạo lại zip:\n  - " + "\n  - ".join(problems[:10]) + extra, "DATA")
    if n_boxes == 0:
        raise NotebookError("Không có hộp nhãn nào trong dữ liệu. Kiểm tra lại vị trí các tệp .txt.", "DATA")

    def is_in(img, names):
        return any(part.lower() in names for part in img.relative_to(raw_dir).parts[:-1])

    pre_val = [it for it in items if is_in(it[0], {"val", "valid", "validation"})]
    pre_train = [it for it in items if is_in(it[0], {"train"})]
    if pre_val and pre_train:  # the zip is already split
        split = {"train": pre_train, "val": pre_val}
    else:
        order = list(range(len(items)))
        random.Random(seed).shuffle(order)
        n_val = max(1, round(len(items) * val_ratio))
        split = {"val": [items[i] for i in order[:n_val]], "train": [items[i] for i in order[n_val:]]}

    counts = {}
    for name, rows in split.items():
        (out_dir / "images" / name).mkdir(parents=True)
        (out_dir / "labels" / name).mkdir(parents=True)
        for k, (img, lines) in enumerate(rows):
            stem = f"{k:05d}_{img.stem}"
            shutil.copyfile(img, out_dir / "images" / name / f"{stem}{img.suffix.lower()}")
            (out_dir / "labels" / name / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
        counts[name] = {"images": len(rows), "boxes": sum(len(lines) for _, lines in rows)}
    data_yaml = out_dir / "data.yaml"
    names_text = "\n".join(f"  {i}: {name}" for i, name in enumerate(CLASSES))
    data_yaml.write_text(f"path: {out_dir.resolve()}\ntrain: images/train\nval: images/val\nnames:\n{names_text}\n", encoding="utf-8")
    print(f"Dữ liệu: {len(images)} ảnh, {n_boxes} hộp `{CLASSES[0]}`. "
          f"train {counts['train']['images']} ảnh / {counts['train']['boxes']} hộp, val {counts['val']['images']} ảnh / {counts['val']['boxes']} hộp.")
    return {"data_yaml": str(data_yaml), "counts": counts, "images": len(images), "boxes": n_boxes}


DATASET = prepare_dataset(get_dataset_zip(DATASET_ZIP_PATH), WORK_DIR / "dataset", SEED)

## Bước 2. Huấn luyện (bạn được sửa tự do)
Ô dưới là **mẫu huấn luyện đơn giản**: bắt đầu từ `yolov8n.pt` rồi fine-tune trên dữ liệu của đội. Bạn tự quyết số epoch, augmentation, nhiều giai đoạn huấn luyện,
dữ liệu bổ sung... Chỉ cần giữ nguyên: **kiến trúc YOLOv8n**, **`imgsz=640`**, **một lớp `GreenSM`**, và `seed=SEED` (mã số đội).
Cuối ô, biến `BEST_WEIGHTS` phải trỏ tới tệp trọng số `.pt` bạn muốn nộp (nếu tự huấn luyện bằng mã riêng, hãy tự gán `BEST_WEIGHTS = "đường_dẫn.pt"`).

In [ ]:
from ultralytics import YOLO

TRAIN_ARGS = dict(
    data=DATASET["data_yaml"],
    imgsz=IMGSZ,            # CỐ ĐỊNH (640): đổi là bài nộp bị từ chối
    epochs=50,
    batch=16,
    seed=SEED,              # seed = mã số đội
    deterministic=True,
    project=str(WORK_DIR / "runs"),
    name="train",
    exist_ok=True,
    # --- Từ đây bạn tự do tinh chỉnh (bỏ dấu # để dùng, xem thêm tài liệu tham số của ultralytics) ---
    # patience=20, optimizer="AdamW", lr0=0.002, cos_lr=True, freeze=10, workers=2,
    # mosaic=1.0, mixup=0.1, copy_paste=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=5, translate=0.1, scale=0.5, fliplr=0.5,
)
if TRAIN_ARGS["imgsz"] != IMGSZ:
    raise NotebookError(f"imgsz phải giữ nguyên {IMGSZ}, nếu không bài nộp sẽ bị từ chối.", "TRAIN")

seed_everything(SEED)
model = YOLO(MODEL_NAME)  # yolov8n.pt: kiến trúc YOLOv8n cố định
model.train(**TRAIN_ARGS)
BEST_WEIGHTS = Path(model.trainer.best)
print("Trọng số tốt nhất:", BEST_WEIGHTS)

### Gợi ý kỹ thuật nâng cao (tùy chọn)
Cuộc thi này chấm **chất lượng dữ liệu** nhiều hơn tinh chỉnh tham số. Một vài hướng để làm trước Bước 2 hoặc kết hợp trong Bước 2:

- **Phân tích lỗi:** `model.val(data=..., plots=True)` và xem ảnh dự đoán sai (xe ban đêm, xe nhỏ, xe bị che, xe giống GreenSM) để biết cần thu thập hoặc sửa nhãn ở đâu.
- **Active learning:** dùng model hiện tại dự đoán trên ảnh chưa gán nhãn, chọn ảnh model kém tự tin để gán và đưa vào vòng sau.
  ```python
  # results = model.predict(source="anh_chua_gan/", conf=0.05, stream=True)
  # khó = [r.path for r in results if len(r.boxes) == 0 or float(r.boxes.conf.min()) < 0.4]
  ```
- **Gán nhãn nháp (pseudo-label) / distillation:** huấn luyện một model lớn hơn (ví dụ `yolov8m.pt`) trên dữ liệu của đội, dùng nó gán nhãn nháp cho nhiều ảnh chưa gán, **người kiểm tra lại**, rồi huấn luyện YOLOv8n trên tập mở rộng.
  ```python
  # teacher = YOLO("duong_dan/teacher_best.pt")
  # teacher.predict(source="anh_chua_gan/", conf=0.5, save_txt=True, save_conf=False)   # nhãn YOLO .txt, cần duyệt lại
  ```
- **Augmentation:** tăng cường theo điều kiện của ảnh kiểm thử (ánh sáng, mờ, nhiễu, xoay, co giãn, `mosaic`, `mixup`, `copy_paste`...). Chia val **trước** khi tăng cường.
- **Dữ liệu tổng hợp / generation:** cắt xe GreenSM từ ảnh đã gán nhãn rồi dán lên nền khác, hoặc sinh ảnh bằng mô hình tạo sinh; giữ đủ ảnh thật để không lệch miền.
- **Ảnh nền (không có xe GreenSM):** thêm ảnh có xe giống GreenSM nhưng không phải (taxi xanh hãng khác...) với tệp nhãn rỗng để model bớt báo nhầm.
- **Tinh chỉnh huấn luyện:** `freeze`, `cos_lr`, `patience`, huấn luyện nhiều giai đoạn (đóng băng rồi mở khóa), chọn checkpoint theo val.

Trọng số cuối cùng bạn muốn nộp phải là mô hình **YOLOv8n 1 lớp** (`.pt`). Gán đường dẫn vào `BEST_WEIGHTS` rồi chạy Bước 3.

## Bước 3. Xuất ONNX, kiểm tra và đóng gói bài nộp (BẮT BUỘC, không sửa)
Ô dưới làm lần lượt:
1. Xuất `BEST_WEIGHTS` ra ONNX với **tham số cố định** do ban tổ chức quy định (`opset=17, simplify=True, dynamic=False, half=False, nms=False, imgsz=640, batch=1`) rồi đặt tên `model.onnx`.
2. **Kiểm tra bằng đúng mã của hệ thống chấm** (YOLOv8n 1 lớp, input/output, fp32, tên lớp, không phải model pretrained gốc, chạy thử một lượt). Nếu không đạt, in đúng thông báo hệ thống sẽ hiển thị và **không tạo zip**.
3. Đóng gói `submission.zip` chứa **đúng một tệp** `model.onnx`, mở lại zip để kiểm tra lần nữa, ghi `run_report.json` (mã số đội, seed, dấu vân tay mô hình).
4. Tải `submission.zip` về máy. **Nộp tệp `submission.zip` này** lên website cuộc thi.

In [ ]:
from ultralytics import YOLO

SUBMISSION = make_submission(BEST_WEIGHTS, WORK_DIR, YOLO, TEAM_ID, SEED)
offer_download([SUBMISSION["zip"], SUBMISSION["report"]])